---
## Cell 1: Imports

In [1]:
!pip install -q pymupdf sentence-transformers qdrant-client bert-score rouge-score sacrebleu scikit-learn tqdm rank_bm25

!pip uninstall torch torchvision torchaudio -y --quiet 2>/dev/null
!pip install torch==2.4.0 torchvision torchaudio \
    --index-url https://download.pytorch.org/whl/cu118 \
    --no-cache-dir --quiet

!pip install -U "transformers==4.48.0" "accelerate==1.2.1" "bitsandbytes==0.45.2" --quiet

import torch, transformers, accelerate, bitsandbytes
print(f"✅ PyTorch {torch.__version__} (CUDA: {torch.version.cuda})")
print(f"✅ Transformers {transformers.__version__}")
print(f"✅ Accelerate {accelerate.__version__}")
print(f"✅ BitsAndBytes {bitsandbytes.__version__}")
print(f"✅ All dependencies installed successfully!")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.0/25.0 MB 73.3 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 389.9/389.9 kB 24.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 8.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 857.7/857.7 MB 221.0 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.2/23.2 MB 334.9 MB/s eta 0:00:00a 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 875.6/875.6 kB 389.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.1/13.1 MB 315.1 MB/s eta 0:00:00a 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 663.9/663.9 MB 308.4 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 417.9/417.9 MB 223.4 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 168.4/168.4 MB 240.0 MB/s eta 0:00:00a 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58

In [2]:
import os
import shutil
from pathlib import Path

SOURCE_DIR = "/kaggle/input/datasets/ismakhrov/pdf-files"
DEST_DIR = "/kaggle/working/pdfs"

os.makedirs(DEST_DIR, exist_ok=True)

print(f"📂 Scanning: {SOURCE_DIR}")
print(f"📁 Destination: {DEST_DIR}\n")

copied_count = 0
duplicate_count = 0

for pdf_path in Path(SOURCE_DIR).rglob("*.pdf"):
    dest_path = Path(DEST_DIR) / pdf_path.name

    if dest_path.exists():
        duplicate_count += 1

    shutil.copy2(pdf_path, dest_path)
    copied_count += 1

print(f"✅ Done! Copied {copied_count} PDF(s).")
print(f"🔍 Duplicates: {duplicate_count}")
print(f"📂 Verify with: !ls -lh {DEST_DIR}")

📂 Scanning: /kaggle/input/datasets/ismakhrov/pdf-files
📁 Destination: /kaggle/working/pdfs

✅ Done! Copied 500 PDF(s).
🔍 Duplicates: 0
📂 Verify with: !ls -lh /kaggle/working/pdfs


In [3]:
import hashlib
import json
import logging
import math
import os
import random
import re
import time
from collections import defaultdict
from pathlib import Path
from typing import Any
from tqdm import tqdm

import numpy as np
import torch
import fitz
import boto3
import sacrebleu
from bert_score import score as bert_score
from rouge_score import rouge_scorer
from sentence_transformers import SentenceTransformer
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from qdrant_client import QdrantClient
from qdrant_client.http import models as rest
from qdrant_client.models import PointStruct
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline

from rank_bm25 import BM25Okapi

print(f"✅ Imports successful")
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

2026-05-09 09:25:03.935472: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1778318704.143874      58 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1778318704.207806      58 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1778318704.676792      58 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1778318704.676852      58 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1778318704.676855      58 computation_placer.cc:177] computation placer alr

✅ Imports successful
PyTorch: 2.4.0+cu118
CUDA available: True
GPU: Tesla T4


---
## Cell 2: Experiment Configuration

In [4]:

EXPERIMENT_NAME = "dense_bge"
PDF_FOLDER = "/kaggle/working/pdfs"

CHUNKING_TYPE = "section_aware"
CHUNK_SIZE = 8192
CHUNK_OVERLAP = 400

EMBEDDING_MODELS = {
    "dense": {
        "model": "sentence-transformers/all-MiniLM-L6-v2",
        "vector_size": 384,
        "field_name": "dense_embedding"
    },
    "mpnet": {
        "model": "sentence-transformers/all-mpnet-base-v2",
        "vector_size": 768,
        "field_name": "dense_mpnet"
    },
    "e5_multilingual": {
        "model": "intfloat/multilingual-e5-large",
        "vector_size": 1024,
        "field_name": "dense_e5"
    },
    "qwen": {
        "model": "Alibaba-NLP/gte-Qwen2-1.5B-instruct",
        "vector_size": 1536,
        "field_name": "gte_qwen2"
    },
    "bge_m3": {
        "model": "BAAI/bge-m3",
        "vector_size": 1024,
        "field_name": "bge_m3"
    },
    "jina_v3": {
        "model": "jinaai/jina-embeddings-v3",
        "vector_size": 1024,
        "field_name": "jina_v3"
    },
    "specter2": {
        "model": "allenai/specter2_base",
        "vector_size": 768,
        "field_name": "specter2"
    },
    "scibert": {
        "model": "allenai/scibert_scivocab_uncased",
        "vector_size": 768,
        "field_name": "scibert"
    }
}
DEFAULT_EMBEDDING = "bge_m3"
EMBEDDING_BATCH_SIZE = 100

RETRIEVER_CONFIG = {
    "type": "dense",
    "top_k": 7,
    
    "dense": {
        "normalize_embeddings": True
    },
    
    "sparse": {
        "method": "bm25",
        "use_qdrant_corpus": True,
        "tfidf_params": {
            "lowercase": True,
            "stop_words": "english",
            "ngram_range": [1, 2],
            "min_df": 1,
            "max_df": 0.9
        },
        "bm25_params": {
            "k1": 1.2,
            "b": 0.75,
            "epsilon": 0.25
        }
    },
    
    "hybrid": {
        "fusion_method": "weighted_sum",
        "dense_weight": 0.7,
        "sparse_weight": 0.3,
        "normalize_scores": True,
        "deduplicate": True,
        "rank_fusion_k": 100
    },
    
    "reranker": {
        "enabled": False,
        "type": "cross_encoder",
        "cross_encoder": {
            "model_name": "cross-encoder/ms-marco-MiniLM-L-6-v2",
            "device": "auto",
            "max_length": 512,
            "batch_size": 32
        }
    }
}

QDRANT_HOST = "localhost"
QDRANT_PORT = 6333
VECTOR_SIZE = 384
QDRANT_BATCH_SIZE = 512

TEST_DATA_SIZE_PERCENT = 10
MAX_QUESTIONS_PER_PAPER = 3
LLM_TEST_DATA_MODEL = "Qwen/Qwen2.5-7B-Instruct"

GENERATOR_MODEL = "Qwen/Qwen2.5-3B-Instruct"
GENERATOR_MAX_TOKENS = 256
BERTSCORE_MODEL = "roberta-base"

USE_LLM_JUDGE = True
LLM_JUDGE_MODEL = "Qwen/Qwen2.5-7B-Instruct"

BASE_DIR = Path("/kaggle/working")
RESULTS_DIR = BASE_DIR / "results" / EXPERIMENT_NAME
CHUNKS_FILE = RESULTS_DIR / "chunks.jsonl"
EMBEDDINGS_FILE = RESULTS_DIR / "embeddings.jsonl"
TEST_DATA_FILE = RESULTS_DIR / "test_data.jsonl"
RESULTS_FILE = RESULTS_DIR / f"{EXPERIMENT_NAME}.json"

RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print(f"Experiment: {EXPERIMENT_NAME}")
print(f"Results directory: {RESULTS_DIR}")
print(f"PDF folder: {PDF_FOLDER}")
print(f"\n=== Configuration ===")
print(f"  Chunking: {CHUNKING_TYPE} ({CHUNK_SIZE} size, {CHUNK_OVERLAP} overlap)")
print(f"  Embeddings: {list(EMBEDDING_MODELS.keys())} (default: {DEFAULT_EMBEDDING})")
print(f"  Retriever: {RETRIEVER_CONFIG['type']} (top_k={RETRIEVER_CONFIG['top_k']})")
print(f"  Re-ranker: {'Enabled' if RETRIEVER_CONFIG['reranker']['enabled'] else 'Disabled'}")
print(f"  Qdrant: {QDRANT_HOST}:{QDRANT_PORT}, vector_size={VECTOR_SIZE}")
print(f"  Test data: {TEST_DATA_SIZE_PERCENT}%, {MAX_QUESTIONS_PER_PAPER} questions/paper")
print(f"  Generator: {GENERATOR_MODEL}")
print(f"  LLM Judge: {'Enabled' if USE_LLM_JUDGE else 'Disabled'}")

Experiment: dense_bge
Results directory: /kaggle/working/results/dense_bge
PDF folder: /kaggle/working/pdfs

=== Configuration ===
  Chunking: section_aware (8192 size, 400 overlap)
  Embeddings: ['dense', 'mpnet', 'e5_multilingual', 'qwen', 'bge_m3', 'jina_v3', 'specter2', 'scibert'] (default: bge_m3)
  Retriever: dense (top_k=7)
  Re-ranker: Disabled
  Qdrant: localhost:6333, vector_size=384
  Test data: 10%, 3 questions/paper
  Generator: Qwen/Qwen2.5-3B-Instruct
  LLM Judge: Enabled


---
## Cell 3: Utility Functions

In [5]:
logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s")
logger = logging.getLogger(__name__)


def save_results(data: dict, filepath: Path):
    """Save experiment results to JSON file."""
    filepath.parent.mkdir(parents=True, exist_ok=True)
    with open(filepath, "w", encoding="utf-8") as f:
        json.dump(data, f, indent=2, ensure_ascii=False)
    logger.info(f"Results saved to {filepath}")


def load_jsonl(filepath: Path) -> list[dict]:
    """Load JSONL file into list of dicts."""
    data = []
    with open(filepath, "r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                data.append(json.loads(line))
    return data


def save_jsonl(data: list[dict], filepath: Path):
    """Save list of dicts to JSONL file."""
    with open(filepath, "w", encoding="utf-8") as f:
        for item in data:
            f.write(json.dumps(item, ensure_ascii=False) + "\n")
    logger.info(f"Saved {len(data)} items to {filepath}")


def load_embeddings_for_model(model_config: dict, embeddings_data: list[dict]) -> list[dict]:
    """Load embeddings for a specific model configuration."""
    field_name = model_config["field_name"]
    
    filtered_embeddings = []
    for record in embeddings_data:
        if field_name in record:
            new_record = record.copy()
            new_record["embedding"] = new_record.pop(field_name)
            filtered_embeddings.append(new_record)
    
    return filtered_embeddings


print("✅ Utility functions defined")

✅ Utility functions defined


---
## Cell 4: PDF Processing & Chunking

In [6]:

SECTION_PATTERNS = {
    "Abstract": [r"^abstract\s*[:.]?\s*$", r"^\s*abstract\s*$", r"^abstract\s+"],
    "Introduction": [
        r"^1\s+introduction\s*$",
        r"^introduction\s*[:.]?\s*$",
        r"^\s*introduction\s*$",
    ],
    "Related Work": [
        r"^related\s+work\s*[:.]?\s*$",
        r"^previous\s+work\s*$",
        r"^literature\s+review\s*$",
    ],
    "Background": [r"^background\s*[:.]?\s*$", r"^\s*background\s*$"],
    "Methodology": [r"^method(ology)?\s*[:.]?\s*$", r"^approach\s*$", r"^proposed\s+method\s*$"],
    "Methods": [r"^methods?\s*[:.]?\s*$"],
    "Model": [r"^model\s*[:.]?\s*$", r"^architecture\s*$", r"^framework\s*$"],
    "Data": [r"^data\s*[:.]?\s*$", r"^dataset(s)?\s*$", r"^data\s+collection\s*$"],
    "Experiments": [r"^experiments?\s*$", r"^evaluation\s*$", r"^experimental\s+setup\s*$"],
    "Results": [r"^results\s*[:.]?\s*$", r"^findings\s*$", r"^empirical\s+results\s*$"],
    "Discussion": [
        r"^discussion\s*[:.]?\s*$",
        r"^analysis\s*$",
        r"^results\s+and\s+discussion\s*$",
    ],
    "Conclusion": [r"^conclusion\s*[:.]?\s*$", r"^future\s+work\s*$", r"^summary\s*$"],
    "References": [r"^references\s*[:.]?\s*$", r"^bibliography\s*$", r"^works\s+cited\s*$"],
    "Appendix": [r"^appendix\s*[:.]?\s*$", r"^supplementary\s+material\s*$", r"^appendices\s*$"],
    "Acknowledgments": [r"^acknowledgments?\s*[:.]?\s*$", r"^acknowledgements?\s*$"],
}

MAIN_TEXT_KEYWORDS = [
    "methods",
    "methodology",
    "approach",
    "model",
    "data",
    "dataset",
    "experiments",
    "results",
    "analysis",
    "discussion",
    "evaluation",
    "related work",
    "preliminaries",
    "framework",
    "architecture",
    "background",
    "motivation",
    "training",
    "inference",
    "implementation",
]

MAX_HEADING_LINE_LENGTH = 100
MAX_HEADING_WORDS = 10
MAX_SHORT_WORD_LENGTH = 3
MIN_HEADING_LENGTH = 2

ALLOWED_LOWERCASE_WORDS = {
    "a",
    "an",
    "the",
    "and",
    "or",
    "but",
    "for",
    "nor",
    "on",
    "at",
    "to",
    "from",
    "by",
    "in",
    "of",
    "with",
    "as",
    "is",
    "are",
    "was",
    "were",
    "be",
    "been",
    "being",
    "have",
    "has",
    "had",
    "do",
    "does",
    "did",
    "will",
    "would",
    "shall",
    "should",
    "may",
    "might",
    "must",
    "can",
    "could",
}

EXCLUSION_PATTERNS = [
    r"^figure\s*\d+",
    r"^fig\.?\s*\d+",
    r"^table\s*\d+",
    r"^tab\.?\s*\d+",
    r"^algorithm\s*\d+",
    r"^equation\s*\d+",
    r"^eq\.?\s*\d+",
]

PDF_ARTIFACTS = [
    r"(?i)\n\s*arxiv:\s*\d+\.\d+(?:v\d+)?\s*\n",
    r"(?i)\n\s*doi:\s*.+\n",
    r"^\s*-\s*\d+\s*-\s*$",
]



def is_excluded_line(line: str) -> bool:
    line_lower = line.lower().strip()
    return any(re.match(pattern, line_lower) for pattern in EXCLUSION_PATTERNS)


def is_heading_line(line: str) -> bool:
    """Heuristic heading detection."""
    if not line or len(line) > MAX_HEADING_LINE_LENGTH:
        return False
    stripped = line.strip()
    if stripped.endswith("!") or stripped.endswith("?") or stripped.endswith(":"):
        return False
    if is_excluded_line(stripped):
        return False
    if re.match(r"^(\d+[\.\)]?|[A-Za-z][\.\)]?|[IVXLCDMivxlcdm]+[\.\)]?)\s+[A-Za-z0-9]", stripped):
        return True
    if stripped.isupper() and len(stripped) > MIN_HEADING_LENGTH:
        return True
    words = stripped.split()
    if 1 <= len(words) <= MAX_HEADING_WORDS:
        for word in words:
            if word.isupper():
                continue
            if (
                word.islower()
                and len(word) <= MAX_SHORT_WORD_LENGTH
                and word.lower() in ALLOWED_LOWERCASE_WORDS
            ):
                continue
            if word and not word[0].isupper():
                return False
        return True
    return False


def classify_header(header_text: str) -> str | None:
    """Classify heading text into a canonical section."""
    header_lower = header_text.lower().strip()
    for section_name, patterns in SECTION_PATTERNS.items():
        for pattern in patterns:
            if re.match(pattern, header_lower, re.IGNORECASE):
                return section_name
    numbered_match = re.match(r"^(\d+(\.\d+)*)\s+(.+)$", header_text.strip())
    if numbered_match:
        heading_text = numbered_match.group(3).lower()
        keyword_map = {
            "related work": "Related Work",
            "background": "Background",
            "method": "Methodology",
            "methods": "Methods",
            "approach": "Methodology",
            "model": "Model",
            "data": "Data",
            "dataset": "Data",
            "experiment": "Experiments",
            "evaluation": "Experiments",
            "result": "Results",
            "results": "Results",
            "findings": "Results",
            "discussion": "Discussion",
            "analysis": "Analysis",
            "conclusion": "Conclusion",
            "summary": "Conclusion",
        }
        for kw, section in keyword_map.items():
            if kw in heading_text:
                return section
        return "Main text"
    for keyword in MAIN_TEXT_KEYWORDS:
        if keyword in header_lower:
            return "Main text"
    return None


def extract_sections_dynamic(text: str, min_section_size: int = 200) -> list[dict]:
    """Extract sections from text by detecting headings."""
    lines = text.split("\n")
    n = len(lines)
    headings = []
    for i, line in enumerate(lines):
        stripped = line.strip()
        if (
            not stripped
            or len(stripped) < MIN_HEADING_LENGTH
            or len(stripped) > MAX_HEADING_LINE_LENGTH
        ):
            continue
        if not is_heading_line(stripped) or is_excluded_line(stripped):
            continue
        classification = classify_header(stripped)
        if classification:
            headings.append((i, classification))
    if not headings:
        return [{"section": "Main text", "text": text.strip(), "start_line": 0, "end_line": n - 1}]
    sections = []
    for idx, (start_idx, section_name) in enumerate(headings):
        end_idx = headings[idx + 1][0] - 1 if idx + 1 < len(headings) else n - 1
        section_text = "\n".join(lines[start_idx : end_idx + 1]).strip()
        if section_text:
            sections.append(
                {
                    "section": section_name,
                    "text": section_text,
                    "start_line": start_idx,
                    "end_line": end_idx,
                }
            )
    merged = [sections[0].copy()]
    for sec in sections[1:]:
        if (
            sec["section"] == merged[-1]["section"]
            and sec["start_line"] == merged[-1]["end_line"] + 1
        ):
            merged[-1]["text"] += "\n\n" + sec["text"]
            merged[-1]["end_line"] = sec["end_line"]
        else:
            merged.append(sec.copy())
    filtered = []
    for sec in merged:
        if len(sec["text"].strip()) < min_section_size and filtered:
            filtered[-1]["text"] += "\n\n" + sec["text"]
        else:
            filtered.append(sec)
    logger.info(f"Extracted {len(filtered)} sections: {[s['section'] for s in filtered]}")
    return filtered



def extract_full_text(pdf_path: str) -> str:
    """Extract full text from PDF using PyMuPDF."""
    doc = fitz.open(pdf_path)
    full_text = ""
    for page in doc:
        full_text += page.get_text("text")
    doc.close()
    return full_text


def preprocess_text(text: str) -> str:
    """Clean and normalize text."""
    text = re.sub(r"-\n\s*", "", text)
    for pattern in PDF_ARTIFACTS:
        text = re.sub(pattern, "\n", text, flags=re.MULTILINE)
    text = re.sub(r"\n{3,}", "\n\n", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()


def _generate_chunk_id(arxiv_id: str, section: str, chunk_idx: int, text: str) -> str:
    """Generate unique chunk ID."""
    hash_input = f"{arxiv_id}:{section}:{chunk_idx}:{text[:100]}"
    hash_value = hashlib.md5(hash_input.encode()).hexdigest()[:8]
    section_clean = section.lower().replace(" ", "_")
    return f"{arxiv_id}_{section_clean}_{chunk_idx:04d}_{hash_value}"


def split_text_recursive(
    text: str, chunk_size: int = 768, chunk_overlap: int = 100, separators: list[str] | None = None
) -> list[str]:
    """Split text into chunks recursively with overlap."""
    if separators is None:
        separators = ["\n\n", "\n", ". ", "! ", "? ", "; ", ", ", " "]
    if len(text) <= chunk_size:
        return [text] if text.strip() else []
    for sep in separators:
        if sep in text:
            parts = text.split(sep)
            chunks = []
            current_chunk = ""
            for part in parts:
                candidate = current_chunk + part + (sep if sep != " " else "")
                if len(candidate) <= chunk_size or not current_chunk:
                    current_chunk = candidate
                else:
                    if current_chunk.strip():
                        chunks.append(current_chunk.rstrip())
                    current_chunk = part + (sep if sep != " " else "")
            if current_chunk.strip():
                chunks.append(current_chunk.rstrip())
            final_chunks = []
            for chunk in chunks:
                if len(chunk) > chunk_size:
                    final_chunks.extend(
                        split_text_recursive(chunk, chunk_size, chunk_overlap, separators)
                    )
                elif chunk.strip():
                    final_chunks.append(chunk)
            return final_chunks
    return [text[i : i + chunk_size] for i in range(0, len(text), chunk_size - chunk_overlap)]


def merge_chunks_by_size(chunks: list[str], min_chunk_size: int, max_chunk_size: int) -> list[str]:
    """Merge undersized chunks with neighbors."""
    if not chunks:
        return []
    merged = []
    buffer = ""
    for chunk_text in chunks:
        chunk_text = chunk_text.strip()
        if not chunk_text:
            continue
        if len(chunk_text) < min_chunk_size:
            buffer = (buffer + "\n\n" + chunk_text).strip() if buffer else chunk_text
        else:
            if buffer:
                if len(buffer) + len(chunk_text) + 2 <= max_chunk_size:
                    chunk_text = buffer + "\n\n" + chunk_text
                else:
                    merged.append(buffer)
                buffer = ""
            merged.append(chunk_text)
    if buffer:
        if merged:
            if len(merged[-1]) + len(buffer) + 2 <= max_chunk_size:
                merged[-1] += "\n\n" + buffer
            else:
                merged.append(buffer)
        else:
            merged.append(buffer)
    return merged


def process_pdfs_to_chunks(
    pdf_folder: str, chunk_size: int, chunk_overlap: int, chunking_type: str = "section_aware"
) -> list[dict]:
    """Process all PDFs in folder and return chunks using specified chunking strategy."""
    pdf_files = list(Path(pdf_folder).glob("*.pdf"))
    if not pdf_files:
        raise ValueError(f"No PDF files found in {pdf_folder}")
    logger.info(f"Found {len(pdf_files)} PDF files (chunking type: {chunking_type})")
    all_chunks = []
    for pdf_path in tqdm(pdf_files):
        try:
            arxiv_id = pdf_path.stem
            arxiv_id_clean = re.sub(r"v\d+$", "", arxiv_id)
            full_text = extract_full_text(str(pdf_path))
            if not full_text.strip():
                logger.warning(f"Empty text from {pdf_path.name}")
                continue
            cleaned_text = preprocess_text(full_text)

            if chunking_type == "section_aware":
                min_section_size = chunk_size // 2
                sections = extract_sections_dynamic(cleaned_text, min_section_size)
                chunk_idx = 0
                for section in sections:
                    section_name = section["section"]
                    section_text = section["text"]
                    if not section_text.strip():
                        continue
                    section_chunks = split_text_recursive(section_text, chunk_size, chunk_overlap)
                    merged = merge_chunks_by_size(section_chunks, chunk_size // 2, chunk_size)
                    for chunk_text in merged:
                        if chunk_text.strip():
                            all_chunks.append(
                                {
                                    "id": _generate_chunk_id(
                                        arxiv_id_clean, section_name, chunk_idx, chunk_text
                                    ),
                                    "text": chunk_text.strip(),
                                    "metadata": {
                                        "arxiv_id": arxiv_id_clean,
                                        "source": pdf_path.name,
                                        "section": section_name,
                                        "chunk_idx": chunk_idx,
                                        "chunk_size": len(chunk_text.strip()),
                                    },
                                }
                            )
                            chunk_idx += 1
            else:
                chunks = split_text_recursive(cleaned_text, chunk_size, chunk_overlap)
                for i, chunk_text in enumerate(chunks):
                    if chunk_text.strip():
                        all_chunks.append(
                            {
                                "id": _generate_chunk_id(
                                    arxiv_id_clean, "full_text", i, chunk_text
                                ),
                                "text": chunk_text.strip(),
                                "metadata": {
                                    "arxiv_id": arxiv_id_clean,
                                    "source": pdf_path.name,
                                    "section": "full_text",
                                    "chunk_idx": i,
                                    "chunk_size": len(chunk_text.strip()),
                                },
                            }
                        )

        except Exception as e:
            logger.error(f"Failed to process {pdf_path.name}: {e}")
            continue

    logger.info(f"Total: {len(all_chunks)} chunks from {len(pdf_files)} papers")
    return all_chunks


print("✅ Chunking functions defined (section-aware + full-text)")

✅ Chunking functions defined (section-aware + full-text)


In [7]:

logger.info("=" * 60)
logger.info("STEP 1: CHUNKING")
logger.info("=" * 60)

chunks = process_pdfs_to_chunks(PDF_FOLDER, CHUNK_SIZE, CHUNK_OVERLAP, CHUNKING_TYPE)
save_jsonl(chunks, CHUNKS_FILE)

logger.info(f"✅ Chunking complete: {len(chunks)} chunks saved to {CHUNKS_FILE}")

2026-05-09 09:25:48,030 - INFO - ============================================================
2026-05-09 09:25:48,031 - INFO - STEP 1: CHUNKING
2026-05-09 09:25:48,032 - INFO - ============================================================
2026-05-09 09:25:48,036 - INFO - Found 500 PDF files (chunking type: section_aware)
100%|██████████| 500/500 [01:00<00:00,  8.24it/s]
2026-05-09 09:26:48,688 - INFO - Total: 4370 chunks from 500 papers
2026-05-09 09:26:49,043 - INFO - Saved 4370 items to /kaggle/working/results/dense_bge/chunks.jsonl
2026-05-09 09:26:49,044 - INFO - ✅ Chunking complete: 4370 chunks saved to /kaggle/working/results/dense_bge/chunks.jsonl


---
## Cell 5: Generate Embeddings

In [8]:


def generate_embeddings_for_model(
    chunks: list[dict],
    model_config: dict,
    batch_size: int = 100,
) -> list[dict]:
    """Generate embeddings with special handling for Jina models."""
    model_name = model_config["model"]
    vector_size = model_config["vector_size"]
    field_name = model_config["field_name"]
    
    logger.info(f"Loading embedding model: {model_name} (vector_size: {vector_size})")
    
    if "jina" in model_name.lower():
        from transformers import AutoModel, AutoTokenizer
        import torch.nn.functional as F
        
        tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
        model = AutoModel.from_pretrained(model_name, trust_remote_code=True)
        model = model.half()
        model = model.to('cuda')
        model.eval()
        
        def encode_texts(texts):
            inputs = tokenizer(texts, padding=True, truncation=True, 
                              return_tensors="pt", max_length=512)
            inputs = {k: v.to('cuda') for k, v in inputs.items()}
            with torch.no_grad():
                outputs = model(**inputs)
            embeddings = outputs.last_hidden_state.mean(dim=1)
            return F.normalize(embeddings, p=2, dim=1).cpu().numpy()
    else:
        model = SentenceTransformer(model_name)
        model = model.half()
        model = model.to('cuda')
        encode_texts = lambda texts: model.encode(texts, show_progress_bar=False, 
                                                  convert_to_numpy=True,
                                                  normalize_embeddings=True)
    
    embeddings_with_chunks = []
    
    for i in tqdm(range(0, len(chunks), batch_size)):
        batch = chunks[i : i + batch_size]
        texts = [c["text"] for c in batch]
        
        embs = encode_texts(texts)
        
        for chunk, emb in zip(batch, embs, strict=False):
            chunk_with_emb = chunk.copy()
            chunk_with_emb[field_name] = emb.tolist()
            embeddings_with_chunks.append(chunk_with_emb)
    
    logger.info(f"Generated {len(embeddings_with_chunks)} embeddings for {model_name}")
    return embeddings_with_chunks

def generate_all_embeddings(
    chunks: list[dict],
    embedding_models: dict,
    default_model: str,
    batch_size: int = 100,
) -> dict[str, list[dict]]:
    """Generate embeddings for all configured models."""
    all_embeddings = {}
    
    for model_name, model_config in embedding_models.items():
        if model_name != default_model:
            continue

        logger.info(f"Generating embeddings for {model_name}...")
        
        embeddings = generate_embeddings_for_model(
            chunks, model_config, batch_size
        )
        
        model_filename = f"embeddings_{model_name}.jsonl"
        model_filepath = RESULTS_DIR / model_filename
        save_jsonl(embeddings, model_filepath)
        
        all_embeddings[model_name] = embeddings
        logger.info(f"Saved {len(embeddings)} embeddings to {model_filepath}")
    
    return all_embeddings


print("✅ Enhanced embedding functions defined")

✅ Enhanced embedding functions defined


In [9]:

logger.info("=" * 60)
logger.info("STEP 2: EMBEDDINGS")
logger.info("=" * 60)

all_embeddings = generate_all_embeddings(
    chunks,
    EMBEDDING_MODELS,
    DEFAULT_EMBEDDING,
    EMBEDDING_BATCH_SIZE
)

default_embeddings = load_embeddings_for_model(
    EMBEDDING_MODELS[DEFAULT_EMBEDDING],
    all_embeddings[DEFAULT_EMBEDDING]
)
save_jsonl(default_embeddings, EMBEDDINGS_FILE)

logger.info(f"✅ Embeddings complete: generated for {len(all_embeddings)} models")
logger.info(f"Default model ({DEFAULT_EMBEDDING}): {len(default_embeddings)} embeddings saved to {EMBEDDINGS_FILE}")

2026-05-09 09:26:56,338 - INFO - ============================================================
2026-05-09 09:26:56,340 - INFO - STEP 2: EMBEDDINGS
2026-05-09 09:26:56,340 - INFO - ============================================================
2026-05-09 09:26:56,341 - INFO - Generating embeddings for bge_m3...
2026-05-09 09:26:56,342 - INFO - Loading embedding model: BAAI/bge-m3 (vector_size: 1024)
2026-05-09 09:26:56,346 - INFO - Use pytorch device_name: cuda:0
2026-05-09 09:26:56,347 - INFO - Load pretrained SentenceTransformer: BAAI/bge-m3


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/54.0 [00:00<?, ?B/s]

...

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

100%|██████████| 44/44 [17:40<00:00, 24.10s/it]
2026-05-09 09:45:03,549 - INFO - Generated 4370 embeddings for BAAI/bge-m3
2026-05-09 09:45:06,823 - INFO - Saved 4370 items to /kaggle/working/results/dense_bge/embeddings_bge_m3.jsonl
2026-05-09 09:45:06,824 - INFO - Saved 4370 embeddings to /kaggle/working/results/dense_bge/embeddings_bge_m3.jsonl
2026-05-09 09:45:10,233 - INFO - Saved 4370 items to /kaggle/working/results/dense_bge/embeddings.jsonl
2026-05-09 09:45:10,234 - INFO - ✅ Embeddings complete: generated for 1 models
2026-05-09 09:45:10,234 - INFO - Default model (bge_m3): 4370 embeddings saved to /kaggle/working/results/dense_bge/embeddings.jsonl


---
## Cell 6: Setup Qdrant (Local)

In [10]:

class LocalQdrantManager:
    """Manage Qdrant collection setup and data ingestion."""

    def __init__(
        self,
        host: str,
        port: int | None,
        collection_name: str,
        vector_size: int,
        batch_size: int = 512,
        in_memory: bool = False,
    ):
        self.collection_name = collection_name
        self.vector_size = vector_size
        self.batch_size = batch_size
        if in_memory:
            self.client = QdrantClient(":memory:")
            logger.info("Using in-memory Qdrant client")
        else:
            self.client = QdrantClient(host=host, port=port)

    def create_collection(self):
        """Create or recreate collection."""
        if self.client.collection_exists(self.collection_name):
            logger.info(f"Collection '{self.collection_name}' already exists, recreating")
            self.client.delete_collection(self.collection_name)

        self.client.create_collection(
            collection_name=self.collection_name,
            vectors_config=rest.VectorParams(
                size=self.vector_size,
                distance=rest.Distance.COSINE,
                on_disk=True,
            ),
            hnsw_config=rest.HnswConfigDiff(m=16, ef_construct=100),
            optimizers_config=rest.OptimizersConfigDiff(indexing_threshold=0),
        )
        logger.info(f"Collection '{self.collection_name}' created")

    def add_data(self, embeddings: list[dict]) -> int:
        """Ingest embeddings into Qdrant."""
        batch = []
        point_id = 0

        for i, record in enumerate(embeddings):
            point = PointStruct(
                id=point_id,
                vector=record["embedding"],
                payload={"text": record["text"], "chunk_id": record["id"], **record["metadata"]},
            )
            batch.append(point)
            point_id += 1

            if len(batch) >= self.batch_size:
                self.client.upsert(collection_name=self.collection_name, points=batch)
                batch = []

        if batch:
            self.client.upsert(collection_name=self.collection_name, points=batch)

        self.client.update_collection(
            collection_name=self.collection_name,
            optimizer_config=rest.OptimizersConfigDiff(indexing_threshold=20000),
        )
        logger.info(f"Total {point_id} points inserted")

        return point_id

    def setup(self, embeddings: list[dict]) -> int:
        """Full setup: create + ingest."""
        self.create_collection()
        return self.add_data(embeddings)


print("✅ Qdrant functions defined")

✅ Qdrant functions defined


In [11]:

import subprocess
import time

QDRANT_IN_MEMORY = True
try:
    test_client = QdrantClient(host="localhost", port=6333, timeout=5)
    test_client.get_collections()
    logger.info("✅ Qdrant is already running on localhost:6333")
except Exception:
    logger.info("Qdrant not running on localhost:6333, switching to in-memory mode")
    QDRANT_HOST = ":memory:"
    QDRANT_PORT = None
    QDRANT_IN_MEMORY = True
    logger.info("✅ Using in-memory Qdrant client (data won't persist)")

print("✅ Qdrant client ready")

2026-05-09 09:45:21,360 - INFO - Qdrant not running on localhost:6333, switching to in-memory mode
2026-05-09 09:45:21,361 - INFO - ✅ Using in-memory Qdrant client (data won't persist)


✅ Qdrant client ready


/usr/local/lib/python3.12/dist-packages/qdrant_client/qdrant_remote.py:288: UserWarning: Failed to obtain server version. Unable to check client-server compatibility. Set check_compatibility=False to skip version check.
  show_warning(


In [12]:

logger.info("=" * 60)
logger.info("STEP 3: QDRANT SETUP")
logger.info("=" * 60)

vector_size = EMBEDDING_MODELS[DEFAULT_EMBEDDING]["vector_size"]

qdrant_manager = LocalQdrantManager(
    host=QDRANT_HOST,
    port=QDRANT_PORT,
    collection_name=EXPERIMENT_NAME,
    vector_size=vector_size,
    batch_size=QDRANT_BATCH_SIZE,
    in_memory=QDRANT_IN_MEMORY,
)

num_points = qdrant_manager.setup(default_embeddings)
logger.info(f"✅ Qdrant setup complete: {num_points} points in collection '{EXPERIMENT_NAME}'")

2026-05-09 09:45:25,110 - INFO - ============================================================
2026-05-09 09:45:25,111 - INFO - STEP 3: QDRANT SETUP
2026-05-09 09:45:25,112 - INFO - ============================================================
2026-05-09 09:45:25,121 - INFO - Using in-memory Qdrant client
2026-05-09 09:45:25,122 - INFO - Collection 'dense_bge' created
2026-05-09 09:45:28,941 - INFO - Total 4370 points inserted
2026-05-09 09:45:28,943 - INFO - ✅ Qdrant setup complete: 4370 points in collection 'dense_bge'


---
## Cell 7: Generate Test Data (LLM-based)

In [13]:


def parse_chunk_id(chunk_id: str) -> dict:
    """Parse chunk ID to extract metadata."""
    if not chunk_id:
        return {"arxiv_id": "", "section": "", "chunk_idx": 0, "hash": ""}
    
    parts = chunk_id.split("_")
    if len(parts) < 4:
        return {"arxiv_id": "", "section": "", "chunk_idx": 0, "hash": ""}
    
    hash_part = parts[-1]
    chunk_idx_part = parts[-2]
    section_parts = parts[:-2]
    
    if not (len(hash_part) == 8 and re.match(r"^[a-f0-9]+$", hash_part)):
        return {"arxiv_id": "", "section": "", "chunk_idx": 0, "hash": ""}
    
    if not (len(chunk_idx_part) == 4 and chunk_idx_part.isdigit()):
        return {"arxiv_id": "", "section": "", "chunk_idx": 0, "hash": ""}
    
    if len(section_parts) >= 2:
        arxiv_id = ".".join(section_parts[:-1])
        section = section_parts[-1]
    elif len(section_parts) == 1:
        arxiv_id = section_parts[0]
        section = ""
    else:
        return {"arxiv_id": "", "section": "", "chunk_idx": 0, "hash": ""}
    
    if not arxiv_id or "." not in arxiv_id:
        return {"arxiv_id": "", "section": "", "chunk_idx": 0, "hash": ""}
    
    return {
        "arxiv_id": arxiv_id,
        "section": section,
        "chunk_idx": int(chunk_idx_part),
        "hash": hash_part,
    }


def group_chunks_by_arxiv(chunks: list[dict]) -> dict[str, list[dict]]:
    """Group chunks by arxiv_id."""
    paper_chunks = defaultdict(list)
    for chunk in chunks:
        chunk_id = chunk.get("id", "")
        if not chunk_id:
            continue
        parsed = parse_chunk_id(chunk_id)
        arxiv_id = parsed["arxiv_id"]
        if arxiv_id:
            paper_chunks[arxiv_id].append(chunk)
    return dict(paper_chunks)


def extract_json_from_text(text: str) -> str:
    """Extract JSON from LLM response."""
    if text is None:
        return ""
    
    text = text.strip()
    code_block_pattern = r"```(?:json)?\s*(.*?)\s*```"
    matches = re.findall(code_block_pattern, text, re.DOTALL)
    if matches:
        content = max(matches, key=len).strip()
        text = content
    
    if not text.startswith("{"):
        start = text.find("{")
        if start == -1:
            return ""
        text = text[start:]
    
    brace_count = 0
    in_string = False
    escape_next = False
    result_end = None
    
    for i, char in enumerate(text):
        if escape_next:
            escape_next = False
            continue
        if char == "\\":
            escape_next = True
            continue
        if char == '"':
            in_string = not in_string
            continue
        if in_string:
            continue
        if char == "{":
            brace_count += 1
        elif char == "}":
            brace_count -= 1
            if brace_count == 0:
                result_end = i + 1
                break
    
    if result_end is not None:
        json_str = text[:result_end].strip()
        try:
            json.loads(json_str)
            return json_str
        except json.JSONDecodeError:
            pass
    
    return ""


def generate_qa_for_paper(
    arxiv_id: str,
    chunks: list[dict],
    num_questions: int,
    pipe,
) -> list[dict]:
    """Generate QA pairs for a single paper using LLM."""
    sorted_chunks = sorted(chunks, key=lambda c: c.get("metadata", {}).get("chunk_idx", 0))
    paper_text = "\n\n".join(
        [c.get("text", "").strip() for c in sorted_chunks if c.get("text", "").strip()]
    )
    
    if not paper_text:
        logger.warning(f"Empty text for paper {arxiv_id}")
        return []
    
    prompt = f"""You are an expert academic researcher. Given the following academic paper text,
generate exactly {num_questions} question-answer pairs.

Requirements:
1. Each question should be answerable FROM THE PROVIDED TEXT.
2. Questions should cover different aspects: main contribution, methodology, experiments/results, specific details.
3. Answers must be directly extractable from the text (verbatim or near-verbatim).
4. Include specific details, numbers, names where appropriate.

Output format (strict JSON only):
{{
  "qa_pairs": [
    {{"question": "...", "answer": "..."}},
    {{"question": "...", "answer": "..."}},
    ...
  ]
}}

Paper text (first 12000 characters):
-------------------
{paper_text[:12000]}
-------------------

Generate {num_questions} diverse Q&A pairs:"""
    
    try:
        messages = [
            {
                "role": "system",
                "content": "You are a helpful academic Q&A generator. Respond with valid JSON only.",
            },
            {"role": "user", "content": prompt},
        ]
        
        output = pipe(
            messages,
            max_new_tokens=1024,
            temperature=0.7,
            do_sample=True,
        )
        
        content = output[0]["generated_text"]
        if isinstance(content, list):
            content = content[-1]["content"]
        else:
            assistant_marker = "assistant"
            if "assistant" in str(content):
                parts = content.rsplit("assistant", 1)
                if len(parts) > 1:
                    content = parts[1].strip()
        
        json_str = extract_json_from_text(content)
        if not json_str:
            logger.warning(f"Failed to extract JSON for {arxiv_id}")
            return []
        
        parsed = json.loads(json_str)
        qa_pairs = parsed.get("qa_pairs", [])
        
        if not isinstance(qa_pairs, list):
            return []
        
        qa_pairs = qa_pairs[:num_questions]
        
        all_chunk_ids = [c.get("id", "") for c in sorted_chunks if c.get("id")]
        results = []
        
        for qa in qa_pairs:
            question = qa.get("question", "").strip()
            answer = qa.get("answer", "").strip()
            
            if not question or not answer:
                continue
            
            answer_keywords = set(w for w in answer[:150].lower().split() if len(w) > 3)
            relevant_chunk_ids = []
            
            if answer_keywords:
                keyword_scores = []
                for chunk in sorted_chunks:
                    chunk_text = chunk.get("text", "").strip().lower()
                    chunk_id = chunk.get("id", "")
                    if not chunk_text or not chunk_id:
                        continue
                    chunk_keywords = set(w for w in chunk_text.split() if len(w) > 3)
                    overlap = len(answer_keywords.intersection(chunk_keywords))
                    if overlap > 0:
                        keyword_scores.append((chunk_id, overlap))
                
                if keyword_scores:
                    keyword_scores.sort(key=lambda x: x[1], reverse=True)
                    relevant_chunk_ids = [cid for cid, _ in keyword_scores[:5]]
            
            if not relevant_chunk_ids and all_chunk_ids:
                mid_idx = len(all_chunk_ids) // 2
                window_size = min(5, len(all_chunk_ids))
                start = max(0, mid_idx - window_size // 2)
                end = min(len(all_chunk_ids), start + window_size)
                relevant_chunk_ids = all_chunk_ids[start:end]
            
            results.append(
                {
                    "question": question,
                    "answer": answer,
                    "relevant_chunk_ids": relevant_chunk_ids,
                    "category": "generated",
                }
            )
        
        return results
    
    except Exception as e:
        logger.error(f"Failed to generate QA for {arxiv_id}: {e}")
        import traceback
        logger.error(traceback.format_exc())
        return []


print("✅ Test data generation functions defined")


✅ Test data generation functions defined


In [14]:

logger.info(f"Loading LLM for test data generation: {LLM_TEST_DATA_MODEL}")

from transformers import BitsAndBytesConfig, AutoModelForCausalLM, AutoTokenizer

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=False,
    bnb_4bit_quant_storage=torch.uint8,
    llm_int8_enable_fp32_cpu_offload=False,
)

_tokenizer = AutoTokenizer.from_pretrained(LLM_TEST_DATA_MODEL)
_model = AutoModelForCausalLM.from_pretrained(
    LLM_TEST_DATA_MODEL,
    device_map="auto",
    torch_dtype=torch.float16,
    quantization_config=bnb_config
)

test_data_pipe = pipeline("text-generation", model=_model, tokenizer=_tokenizer)

logger.info(f"✅ LLM loaded with 4-bit quantization: {LLM_TEST_DATA_MODEL}")

2026-05-09 09:45:36,356 - INFO - Loading LLM for test data generation: Qwen/Qwen2.5-7B-Instruct


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

model-00003-of-00004.safetensors:   0%|          | 0.00/3.86G [00:00<?, ?B/s]

model-00004-of-00004.safetensors:   0%|          | 0.00/3.56G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

Device set to use cuda:0
2026-05-09 09:47:41,018 - INFO - ✅ LLM loaded with 4-bit quantization: Qwen/Qwen2.5-7B-Instruct


In [15]:

logger.info("=" * 60)
logger.info("STEP 4: TEST DATA GENERATION")
logger.info("=" * 60)

paper_to_chunks = group_chunks_by_arxiv(chunks)
total_papers = len(paper_to_chunks)
logger.info(f"Total papers: {total_papers}")

paper_ids = list(paper_to_chunks.keys())
random.seed(42)
random.shuffle(paper_ids)

if TEST_DATA_SIZE_PERCENT > 0:
    target_papers = max(1, int(total_papers * TEST_DATA_SIZE_PERCENT / 100))
else:
    target_papers = total_papers

target_paper_ids = paper_ids[:target_papers]
logger.info(f"Target: {target_papers} papers")

all_samples = []
for idx, arxiv_id in enumerate(target_paper_ids, 1):
    logger.info(f"[{idx}/{target_papers}] Processing: {arxiv_id}")

    samples = generate_qa_for_paper(
        arxiv_id=arxiv_id,
        chunks=paper_to_chunks[arxiv_id],
        num_questions=MAX_QUESTIONS_PER_PAPER,
        pipe=test_data_pipe,
    )

    if samples:
        all_samples.extend(samples)
        logger.info(f"  ✅ Generated {len(samples)} Q&A pairs")
    else:
        logger.warning(f"  ⚠️ No Q&A pairs generated")

save_jsonl(all_samples, TEST_DATA_FILE)
logger.info(f"✅ Test data complete: {len(all_samples)} samples saved to {TEST_DATA_FILE}")

2026-05-09 09:47:41,027 - INFO - ============================================================
2026-05-09 09:47:41,028 - INFO - STEP 4: TEST DATA GENERATION
2026-05-09 09:47:41,029 - INFO - ============================================================
2026-05-09 09:47:41,041 - INFO - Total papers: 500
2026-05-09 09:47:41,043 - INFO - Target: 50 papers
2026-05-09 09:47:41,044 - INFO - [1/50] Processing: 2604.03583.main
2026-05-09 09:47:55,728 - INFO -   ✅ Generated 3 Q&A pairs
2026-05-09 09:47:55,729 - INFO - [2/50] Processing: 2604.06409.main
2026-05-09 09:48:07,541 - INFO -   ✅ Generated 3 Q&A pairs
2026-05-09 09:48:07,542 - INFO - [3/50] Processing: 2604.04593.main
2026-05-09 09:48:26,678 - WARNING -   ⚠️ No Q&A pairs generated
2026-05-09 09:48:26,679 - INFO - [4/50] Processing: 2604.04692.main
2026-05-09 09:48:44,048 - INFO -   ✅ Generated 3 Q&A pairs
2026-05-09 09:48:44,049 - INFO - [5/50] Processing: 2604.02669.main
2026-05-09 09:48:59,155 - INFO -   ✅ Generated 3 Q&A pairs
2026-05-

---
## Cell 8: Enhanced Retriever Evaluation

In [30]:

class EnhancedDenseRetriever:
    """Enhanced dense retriever with multiple embedding model support including Jina."""

    def __init__(
        self,
        collection_name: str,
        embedding_model_name: str,
        vector_size: int,
        top_k: int = 5,
        qdrant_client: QdrantClient = None,
        host: str = None,
        port: int = None,
        in_memory: bool = False,
    ):
        self.collection_name = collection_name
        self.embedding_model_name = embedding_model_name
        self.vector_size = vector_size
        self.top_k = top_k
        
        logger.info(f"Loading embedding model: {embedding_model_name}")
        
        if "jina" in embedding_model_name.lower():
            from transformers import AutoModel, AutoTokenizer
            import torch.nn.functional as F
            
            self.tokenizer = AutoTokenizer.from_pretrained(embedding_model_name, trust_remote_code=True)
            self.model = AutoModel.from_pretrained(embedding_model_name, trust_remote_code=True)
            self.model = self.model.half()
            self.model = self.model.to('cuda')
            self.model.eval()
            self.use_sentence_transformer = False
        else:
            self.embedder = SentenceTransformer(embedding_model_name)
            self.use_sentence_transformer = True
        
        if qdrant_client is not None:
            self.client = qdrant_client
        elif in_memory:
            self.client = QdrantClient(":memory:")
        else:
            self.client = QdrantClient(host=host, port=port)

    def retrieve(self, query: str, top_k: int = 0) -> list[dict]:
        k = top_k or self.top_k
        
        if self.use_sentence_transformer:
            query_vector = self.embedder.encode(query, normalize_embeddings=True).tolist()
        else:
            inputs = self.tokenizer(query, padding=True, truncation=True, 
                                   return_tensors="pt", max_length=512)
            inputs = {k: v.to('cuda') for k, v in inputs.items()}
            with torch.no_grad():
                outputs = self.model(**inputs)
            embedding = outputs.last_hidden_state.mean(dim=1)
            embedding = torch.nn.functional.normalize(embedding, p=2, dim=1)
            query_vector = embedding.cpu().numpy()[0].tolist()

        results = self.client.query_points(
            collection_name=self.collection_name,
            query=query_vector,
            limit=k,
            with_payload=True,
        )

        return [
            {"id": hit.payload.get("chunk_id", hit.id), "score": hit.score, "payload": hit.payload}
            for hit in results.points
        ]


class EnhancedSparseRetriever:
    """Enhanced sparse retriever using BM25 and TF-IDF."""
    
    def __init__(
        self,
        collection_name: str,
        method: str = "bm25",
        tfidf_params: dict = None,
        bm25_params: dict = None,
        use_qdrant_corpus: bool = True,
        top_k: int = 5,
        qdrant_client: QdrantClient = None,
    ):
        self.collection_name = collection_name
        self.method = method
        self.use_qdrant_corpus = use_qdrant_corpus
        self.top_k = top_k
        self.client = qdrant_client
        
        self.tfidf_params = tfidf_params or {
            "lowercase": True,
            "stop_words": "english",
            "ngram_range": (1, 2),
            "min_df": 1,
            "max_df": 0.9
        }
        
        self.bm25_params = bm25_params or {
            "k1": 1.2,
            "b": 0.75,
            "epsilon": 0.25
        }
        
        self.vectorizer = TfidfVectorizer(**self.tfidf_params) if method == "tfidf" else None
        self.bm25 = None
        self.corpus_texts = []
        self.is_built = False
    
    def _extract_corpus_from_qdrant(self):
        """Extract document texts from Qdrant collection."""
        if not self.client:
            return []
        
        try:
            response = self.client.scroll(
                collection_name=self.collection_name,
                limit=10000,
                with_payload=True,
                with_vectors=False,
            )
            
            corpus = []
            for point in response[0]:
                text = point.payload.get("text", "")
                if text:
                    corpus.append(text)
            
            logger.info(f"Extracted {len(corpus)} documents from Qdrant")
            return corpus
            
        except Exception as e:
            logger.error(f"Failed to extract corpus from Qdrant: {e}")
            return []
    
    def _build_index(self):
        """Build the sparse index."""
        if self.use_qdrant_corpus and self.client:
            self.corpus_texts = self._extract_corpus_from_qdrant()
        
        if not self.corpus_texts:
            logger.warning("No corpus available for sparse retriever")
            return
        
        if self.method == "tfidf":
            self.vectorizer.fit(self.corpus_texts)
        else:
            tokenized_corpus = []
            for text in self.corpus_texts:
                tokens = re.findall(r'\b\w+\b', text.lower())
                tokens = [token for token in tokens if len(token) > 2]
                tokenized_corpus.append(tokens)
            self.bm25 = BM25Okapi(tokenized_corpus, **self.bm25_params)
        
        self.is_built = True
        logger.info(f"Built {self.method} index for {len(self.corpus_texts)} documents")
    
    def _tokenize_text(self, text: str) -> list[str]:
        """Tokenize text for BM25."""
        text = text.lower()
        tokens = re.findall(r'\b\w+\b', text)
        tokens = [token for token in tokens if len(token) > 2]
        return tokens
    
    def retrieve(self, query: str, top_k: int = 0) -> list[dict]:
        k = top_k or self.top_k
        
        if not self.is_built:
            self._build_index()
        
        if not self.is_built:
            return []
        
        if self.method == "tfidf":
            query_vec = self.vectorizer.transform([query])
            similarities = (query_vec * self.vectorizer.transform(self.corpus_texts).T).toarray()[0]
            
            top_indices = np.argsort(similarities)[::-1][:k]
            
            results = []
            for idx in top_indices:
                if similarities[idx] > 0:
                    results.append({
                        "id": f"tfidf_{idx}",
                        "score": float(similarities[idx]),
                        "payload": {
                            "text": self.corpus_texts[idx],
                            "retriever_type": "tfidf",
                        }
                    })
        else:
            query_tokens = self._tokenize_text(query)
            bm25_scores = self.bm25.get_scores(query_tokens)
            
            top_indices = np.argsort(bm25_scores)[::-1][:k]
            
            results = []
            for idx in top_indices:
                if bm25_scores[idx] > 0:
                    results.append({
                        "id": f"bm25_{idx}",
                        "score": float(bm25_scores[idx]),
                        "payload": {
                            "text": self.corpus_texts[idx],
                            "retriever_type": "bm25",
                        }
                    })
        
        return results


class CrossEncoderReranker:
    """Cross-encoder reranker for re-ranking retrieved documents."""
    
    def __init__(self, model_name: str = "cross-encoder/ms-marco-MiniLM-L-6-v2", max_length: int = 512):
        self.model_name = model_name
        self.max_length = max_length
        
        logger.info(f"Loading cross-encoder model: {model_name}")
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModelForSequenceClassification.from_pretrained(model_name)
        self.model.eval()
    
    def rerank(self, query: str, documents: list[dict], top_k: int = None) -> list[dict]:
        """Re-rank documents based on relevance to the query."""
        if not documents:
            return []
        
        pairs = []
        for doc in documents:
            text = doc.get("payload", {}).get("text", "")
            if text:
                pairs.append(f"{query} [SEP] {text}")
        
        if not pairs:
            return documents
        
        inputs = self.tokenizer(
            pairs,
            truncation=True,
            padding=True,
            max_length=self.max_length,
            return_tensors="pt"
        )
        
        with torch.no_grad():
            outputs = self.model(**inputs)
            scores = torch.softmax(outputs.logits, dim=1)[:, 1].tolist()
        
        scored_docs = []
        for doc, score in zip(documents, scores):
            if score > 0:
                reranked_doc = doc.copy()
                reranked_doc["score"] = score
                reranked_doc["rerank_score"] = score
                scored_docs.append(reranked_doc)
        
        scored_docs.sort(key=lambda x: x["rerank_score"], reverse=True)
        
        k = top_k if top_k is not None else len(scored_docs)
        return scored_docs[:k]


def recall_at_k(retrieved_ids: list, relevant_ids: set, k: int) -> float:
    if not relevant_ids:
        return 0.0
    top_k = set(retrieved_ids[:k])
    return len(top_k & relevant_ids) / len(relevant_ids)


def precision_at_k(retrieved_ids: list, relevant_ids: set, k: int) -> float:
    top_k = set(retrieved_ids[:k])
    if not top_k:
        return 0.0
    return len(top_k & relevant_ids) / k


def accuracy_at_k(retrieved_ids: list, relevant_ids: set, k: int) -> float:
    top_k = set(retrieved_ids[:k])
    return 1.0 if (top_k & relevant_ids) else 0.0


def f1_at_k(retrieved_ids: list, relevant_ids: set, k: int) -> float:
    p = precision_at_k(retrieved_ids, relevant_ids, k)
    r = recall_at_k(retrieved_ids, relevant_ids, k)
    if p + r == 0:
        return 0.0
    return 2 * (p * r) / (p + r)


def mrr_at_k(retrieved_ids: list, relevant_ids: set, k: int) -> float:
    for i, rid in enumerate(retrieved_ids[:k]):
        if rid in relevant_ids:
            return 1.0 / (i + 1)
    return 0.0


def ndcg_at_k(retrieved_ids: list, relevant_ids: set, k: int) -> float:
    if not relevant_ids:
        return 0.0

    relevance = [1 if rid in relevant_ids else 0 for rid in retrieved_ids[:k]]
    if not relevance:
        return 0.0

    dcg = sum(rel / math.log2(i + 2) for i, rel in enumerate(relevance))

    ideal_relevance = sorted(relevance, reverse=True)
    idcg = sum(rel / math.log2(i + 2) for i, rel in enumerate(ideal_relevance))

    if idcg == 0:
        return 0.0
    return dcg / idcg


def evaluate_retriever(test_samples: list[dict], retriever, top_k: int, retriever_name: str) -> dict:
    """Evaluate retriever on test samples."""
    recall_scores, precision_scores, accuracy_scores = [], [], []
    f1_scores, mrr_scores, ndcg_scores = [], [], []

    for i, sample in tqdm(enumerate(test_samples)):
        docs = retriever.retrieve(sample["question"], top_k=top_k)
        retrieved_ids = [doc["id"] for doc in docs]
        relevant_ids = set(sample.get("relevant_chunk_ids", []))

        if not relevant_ids:
            continue

        recall_scores.append(recall_at_k(retrieved_ids, relevant_ids, top_k))
        precision_scores.append(precision_at_k(retrieved_ids, relevant_ids, top_k))
        accuracy_scores.append(accuracy_at_k(retrieved_ids, relevant_ids, top_k))
        f1_scores.append(f1_at_k(retrieved_ids, relevant_ids, top_k))
        mrr_scores.append(mrr_at_k(retrieved_ids, relevant_ids, top_k))
        ndcg_scores.append(ndcg_at_k(retrieved_ids, relevant_ids, top_k))

    def avg(scores):
        return sum(scores) / len(scores) if scores else 0.0

    metrics = {
        "precision_at_k": avg(precision_scores),
        "recall_at_k": avg(recall_scores),
        "accuracy_at_k": avg(accuracy_scores),
        "f1_at_k": avg(f1_scores),
        "mrr_at_k": avg(mrr_scores),
        "ndcg_at_k": avg(ndcg_scores),
    }

    logger.info(f"\n{retriever_name} Metrics (Top-{top_k}):")
    for k, v in metrics.items():
        logger.info(f"  {k}: {v:.4f}")

    return metrics


print("✅ Enhanced retriever evaluation functions defined")

✅ Enhanced retriever evaluation functions defined


In [31]:

logger.info("=" * 60)
logger.info("STEP 5: ENHANCED RETRIEVER EVALUATION")
logger.info("=" * 60)

qdrant_client = qdrant_manager.client
TOP_K = 7

all_retriever_metrics = {}

logger.info("\n--- Evaluating Dense Retriever ---")
dense_retriever = EnhancedDenseRetriever(
    collection_name=EXPERIMENT_NAME,
    embedding_model_name=EMBEDDING_MODELS[DEFAULT_EMBEDDING]["model"],
    vector_size=EMBEDDING_MODELS[DEFAULT_EMBEDDING]["vector_size"],
    top_k=TOP_K,
    qdrant_client=qdrant_client,
)
dense_metrics = evaluate_retriever(all_samples, dense_retriever, TOP_K, "Dense Retriever")
all_retriever_metrics["dense"] = dense_metrics

logger.info("\n--- Evaluating Re-ranker ---")
reranker = CrossEncoderReranker()

initial_docs = dense_retriever.retrieve("sample query", top_k=TOP_K * 2)
reranked_docs = reranker.rerank("sample query", initial_docs, top_k=TOP_K)

class RerankEnhancedRetriever:
    def __init__(self, base_retriever, reranker, top_k):
        self.base_retriever = base_retriever
        self.reranker = reranker
        self.top_k = top_k
    
    def retrieve(self, query, top_k=0):
        k = top_k or self.top_k
        initial_docs = self.base_retriever.retrieve(query, top_k=k * 2)
        return self.reranker.rerank(query, initial_docs, top_k=k)

rerank_retriever = RerankEnhancedRetriever(dense_retriever, reranker, TOP_K)
rerank_metrics = evaluate_retriever(all_samples, rerank_retriever, TOP_K, "Dense + Re-ranker")
all_retriever_metrics["dense_rerank"] = rerank_metrics

logger.info("\n✅ All retrievers evaluated successfully!")
logger.info(f"\nSummary: Evaluated {len(all_retriever_metrics)} retriever configurations")

2026-05-09 10:07:53,668 - INFO - ============================================================
2026-05-09 10:07:53,669 - INFO - STEP 5: ENHANCED RETRIEVER EVALUATION
2026-05-09 10:07:53,670 - INFO - ============================================================
2026-05-09 10:07:53,671 - INFO - 
--- Evaluating Dense Retriever ---
2026-05-09 10:07:53,672 - INFO - Loading embedding model: BAAI/bge-m3
2026-05-09 10:07:53,685 - INFO - Use pytorch device_name: cuda:0
2026-05-09 10:07:53,685 - INFO - Load pretrained SentenceTransformer: BAAI/bge-m3
0it [00:00, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

1it [00:00,  5.54it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

2it [00:00,  7.07it/s]

...

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

NameError: name 'AutoModelForSequenceClassification' is not defined

In [34]:

from rank_bm25 import BM25Okapi
import re

try:
    chunks = load_jsonl(CHUNKS_FILE)
    logger.info(f"Loaded {len(chunks)} chunks")
except:
    logger.error("Chunks not found. Run Cell 4 first.")

logger.info("Building BM25 index from chunks...")

tokenized_chunks = []
for chunk in chunks:
    text = chunk['text'].lower()
    tokens = re.findall(r'\b\w+\b', text)
    tokens = [t for t in tokens if len(t) > 2]
    tokenized_chunks.append(tokens)

bm25_index = BM25Okapi(tokenized_chunks)
logger.info(f"✅ BM25 index built with {len(chunks)} documents")

class SimpleSparseRetriever:
    def __init__(self, chunks, bm25_index):
        self.chunks = chunks
        self.bm25_index = bm25_index
        self.top_k = 10
    
    def retrieve(self, query, top_k=0):
        k = top_k or self.top_k
        
        tokens = re.findall(r'\b\w+\b', query.lower())
        tokens = [t for t in tokens if len(t) > 2]
        
        scores = self.bm25_index.get_scores(tokens)
        
        top_indices = sorted(range(len(scores)), 
                            key=lambda i: scores[i], 
                            reverse=True)[:k]
        
        results = []
        for idx in top_indices:
            if scores[idx] > 0:
                results.append({
                    "id": self.chunks[idx]["id"],
                    "score": float(scores[idx]),
                    "payload": {
                        "text": self.chunks[idx]["text"],
                        **self.chunks[idx]["metadata"]
                    }
                })
        
        return results

sparse_retriever_bm25 = SimpleSparseRetriever(chunks, bm25_index)

test_query = "What is machine learning?"
results = sparse_retriever_bm25.retrieve(test_query, top_k=3)

print(f"\n🔍 Testing sparse retriever with query: '{test_query}'")
for i, r in enumerate(results, 1):
    print(f"\n{i}. Score: {r['score']:.4f}")
    print(f"   Chunk ID: {r['id']}")
    print(f"   Preview: {r['payload']['text'][:200]}...")


2026-05-09 10:08:34,648 - INFO - Loaded 4370 chunks
2026-05-09 10:08:34,649 - INFO - Building BM25 index from chunks...
2026-05-09 10:08:38,136 - INFO - ✅ BM25 index built with 4370 documents



🔍 Testing sparse retriever with query: 'What is machine learning?'

1. Score: 5.5166
   Chunk ID: 2604.07591_main_text_0000_30e363a3
   Preview: Measurement Error Labeling From Ground Truth to Measurement: A Statistical Framework for Human Labeling Robert Chew* rchew@rti.org Stephanie Eckman† steph@umd.edu Christoph Kern‡§ christoph.kern@lmu.d...

2. Score: 5.4459
   Chunk ID: 2604.07591_main_text_0007_d3801386
   Preview: Future work could perform simliar decompositions on a wider variety of tasks that vary in subjectivity, domain expertise, and stakes. Third, though we model item features (e.g., ambiguity) and include...

3. Score: 5.4040
   Chunk ID: 2604.07036_main_text_0004_a5e698e6
   Preview: Aakanksha Chowdhery, Sharan Narang, Jacob Devlin, Maarten Bosma, Gaurav Mishra, Adam Roberts, Paul Barham, Hyung Won Chung, Charles Sutton, Sebastian Gehrmann, et al. PaLM: Scaling language modeling w...


In [35]:
from sklearn.feature_extraction.text import TfidfVectorizer
import numpy as np
import re

chunks = load_jsonl(CHUNKS_FILE)
corpus_texts = [chunk['text'] for chunk in chunks]

tfidf_vectorizer = TfidfVectorizer(
    lowercase=True,
    stop_words='english',
    ngram_range=(1, 2),
    max_df=0.9,
    min_df=2,
    sublinear_tf=True
)

logger.info("Building TF-IDF matrix...")
tfidf_matrix = tfidf_vectorizer.fit_transform(corpus_texts)
logger.info(f"✅ TF-IDF matrix shape: {tfidf_matrix.shape}")

class TfidfRetriever:
    def __init__(self, chunks, vectorizer, tfidf_matrix):
        self.chunks = chunks
        self.vectorizer = vectorizer
        self.tfidf_matrix = tfidf_matrix
        self.top_k = 10
    
    def retrieve(self, query, top_k=0):
        k = top_k or self.top_k
        
        query_vec = self.vectorizer.transform([query])
        
        from sklearn.metrics.pairwise import cosine_similarity
        similarities = cosine_similarity(query_vec, self.tfidf_matrix).flatten()
        
        top_indices = np.argsort(similarities)[::-1][:k]
        
        results = []
        for idx in top_indices:
            if similarities[idx] > 0:
                results.append({
                    "id": self.chunks[idx]["id"],
                    "score": float(similarities[idx]),
                    "payload": {
                        "text": self.chunks[idx]["text"],
                        **self.chunks[idx]["metadata"]
                    }
                })
        
        return results

tfidf_retriever = TfidfRetriever(chunks, tfidf_vectorizer, tfidf_matrix)

test_query = "neural network architecture"
results = tfidf_retriever.retrieve(test_query, top_k=5)

print(f"\n🔍 TF-IDF Results for: '{test_query}'")
for i, r in enumerate(results, 1):
    print(f"\n{i}. Score: {r['score']:.4f}")
    print(f"   Chunk: {r['payload']['section'] if 'section' in r['payload'] else 'unknown'}")
    print(f"   Preview: {r['payload']['text'][:150]}...")

2026-05-09 10:08:40,653 - INFO - Building TF-IDF matrix...
2026-05-09 10:08:49,708 - INFO - ✅ TF-IDF matrix shape: (4370, 423109)



🔍 TF-IDF Results for: 'neural network architecture'

1. Score: 0.0793
   Chunk: Main text
   Preview: In: Intelligent, Secure, and Dependable Systems in Distributed and Cloud Environments: First International Conference, ISDDC 2017, Vancouver, BC, Cana...

2. Score: 0.0749
   Chunk: Main text
   Preview: Preprint. This article has been accepted at the 19th IEEE International Conference on Software Testing, Verification and Validation (ICST) 2026. This ...

3. Score: 0.0736
   Chunk: Main text
   Preview: 1) Seed Selection: To ensure domain diversity and reduce bias, we selected 381 seed URLs from five distinct categories: News Media (e.g., BBC, CNN), E...

4. Score: 0.0696
   Chunk: Main text
   Preview: Kim, S. Chang, and N. Kwak, “Pqk: model compression via pruning, quantization, and knowledge distillation,” arXiv preprint arXiv:2106.14681, 2021. [10...

5. Score: 0.0571
   Chunk: Main text
   Preview: • We introduce and quantify a tokenization-level asymmetry for skin tones in LLMs

In [36]:

logger.info("=" * 60)
logger.info("STEP 5: ENHANCED RETRIEVER EVALUATION")
logger.info("=" * 60)

all_retriever_metrics = {}
TOP_K = 7

logger.info("\n--- Evaluating Sparse Retriever (BM25) ---")

sparse_bm25_metrics = evaluate_retriever(all_samples, sparse_retriever_bm25, TOP_K, "Sparse Retriever (BM25)")
all_retriever_metrics["sparse_bm25"] = sparse_bm25_metrics

logger.info("\n--- Evaluating Sparse Retriever (TF-IDF) ---")

sparse_tfidf_metrics = evaluate_retriever(all_samples, tfidf_retriever, TOP_K, "Sparse Retriever (TF-IDF)")
all_retriever_metrics["sparse_tfidf"] = sparse_tfidf_metrics

logger.info("\n✅ All retrievers evaluated successfully!")
logger.info(f"\nSummary: Evaluated {len(all_retriever_metrics)} retriever configurations")

2026-05-09 10:08:49,949 - INFO - ============================================================
2026-05-09 10:08:49,951 - INFO - STEP 5: ENHANCED RETRIEVER EVALUATION
2026-05-09 10:08:49,952 - INFO - ============================================================
2026-05-09 10:08:49,953 - INFO - 
--- Evaluating Sparse Retriever (BM25) ---
126it [00:02, 44.11it/s]
2026-05-09 10:08:52,814 - INFO - 
Sparse Retriever (BM25) Metrics (Top-7):
2026-05-09 10:08:52,815 - INFO -   precision_at_k: 0.4512
2026-05-09 10:08:52,816 - INFO -   recall_at_k: 0.6403
2026-05-09 10:08:52,816 - INFO -   accuracy_at_k: 0.8810
2026-05-09 10:08:52,817 - INFO -   f1_at_k: 0.5288
2026-05-09 10:08:52,818 - INFO -   mrr_at_k: 0.8267
2026-05-09 10:08:52,818 - INFO -   ndcg_at_k: 0.8197
2026-05-09 10:08:52,819 - INFO - 
--- Evaluating Sparse Retriever (TF-IDF) ---
126it [00:27,  4.63it/s]
2026-05-09 10:09:20,014 - INFO - 
Sparse Retriever (TF-IDF) Metrics (Top-7):
2026-05-09 10:09:20,015 - INFO -   precision_at_k: 0.4751

In [37]:

class HybridRetriever:
    """Hybrid retriever combining dense and sparse retrieval with multiple fusion methods."""
    
    def __init__(
        self,
        dense_retriever,
        sparse_retriever,
        fusion_method: str = "weighted_sum",
        dense_weight: float = 0.7,
        sparse_weight: float = 0.3,
        normalize_scores: bool = True,
        rrf_k: int = 60,
        top_k: int = 7,
    ):
        self.dense_retriever = dense_retriever
        self.sparse_retriever = sparse_retriever
        self.fusion_method = fusion_method
        self.dense_weight = dense_weight
        self.sparse_weight = sparse_weight
        self.normalize_scores = normalize_scores
        self.rrf_k = rrf_k
        self.top_k = top_k
    
    def _normalize_scores(self, results: list[dict]) -> list[dict]:
        """Min-max normalize scores to [0, 1] range."""
        if not results:
            return results
        
        scores = [r["score"] for r in results]
        min_score = min(scores)
        max_score = max(scores)
        
        if max_score == min_score:
            for r in results:
                r["normalized_score"] = 0.5
            return results
        
        for r in results:
            r["normalized_score"] = (r["score"] - min_score) / (max_score - min_score)
        
        return results
    
    def _weighted_sum_fusion(self, dense_results, sparse_results) -> list[dict]:
        """Fuse scores using weighted sum of normalized scores."""
        if self.normalize_scores:
            dense_results = self._normalize_scores(dense_results)
            sparse_results = self._normalize_scores(sparse_results)
            dense_key = "normalized_score"
            sparse_key = "normalized_score"
        else:
            dense_key = "score"
            sparse_key = "score"
        
        dense_dict = {r["id"]: r[dense_key] for r in dense_results}
        sparse_dict = {r["id"]: r[sparse_key] for r in sparse_results}
        
        all_ids = set(dense_dict.keys()) | set(sparse_dict.keys())
        
        fused = []
        for doc_id in all_ids:
            dense_score = dense_dict.get(doc_id, 0) * self.dense_weight
            sparse_score = sparse_dict.get(doc_id, 0) * self.sparse_weight
            fused_score = dense_score + sparse_score
            
            payload = None
            for r in dense_results + sparse_results:
                if r["id"] == doc_id:
                    payload = r.get("payload", {})
                    break
            
            fused.append({
                "id": doc_id,
                "score": fused_score,
                "payload": payload,
                "dense_score": dense_dict.get(doc_id, 0),
                "sparse_score": sparse_dict.get(doc_id, 0),
            })
        
        fused.sort(key=lambda x: x["score"], reverse=True)
        return fused
    
    def _rrf_fusion(self, dense_results, sparse_results) -> list[dict]:
        """
        Reciprocal Rank Fusion (RRF).
        Standard formula: score = sum(1 / (k + rank))
        """
        dense_ranks = {r["id"]: idx + 1 for idx, r in enumerate(dense_results)}
        sparse_ranks = {r["id"]: idx + 1 for idx, r in enumerate(sparse_results)}
        
        all_ids = set(dense_ranks.keys()) | set(sparse_ranks.keys())
        
        fused = []
        for doc_id in all_ids:
            rrf_score = 0.0
            
            if doc_id in dense_ranks:
                rrf_score += 1.0 / (self.rrf_k + dense_ranks[doc_id])
            if doc_id in sparse_ranks:
                rrf_score += 1.0 / (self.rrf_k + sparse_ranks[doc_id])
            
            payload = None
            for r in dense_results + sparse_results:
                if r["id"] == doc_id:
                    payload = r.get("payload", {})
                    break
            
            fused.append({
                "id": doc_id,
                "score": rrf_score,
                "payload": payload,
                "dense_rank": dense_ranks.get(doc_id, None),
                "sparse_rank": sparse_ranks.get(doc_id, None),
            })
        
        fused.sort(key=lambda x: x["score"], reverse=True)
        return fused
    
    def _rank_fusion_with_weights(self, dense_results, sparse_results) -> list[dict]:
        """
        Weighted Rank Fusion.
        Combines ranks with weights: score = w_dense/(k + rank_dense) + w_sparse/(k + rank_sparse)
        """
        dense_ranks = {r["id"]: idx + 1 for idx, r in enumerate(dense_results)}
        sparse_ranks = {r["id"]: idx + 1 for idx, r in enumerate(sparse_results)}
        
        all_ids = set(dense_ranks.keys()) | set(sparse_ranks.keys())
        
        fused = []
        for doc_id in all_ids:
            rank_score = 0.0
            
            if doc_id in dense_ranks:
                rank_score += self.dense_weight / (self.rrf_k + dense_ranks[doc_id])
            if doc_id in sparse_ranks:
                rank_score += self.sparse_weight / (self.rrf_k + sparse_ranks[doc_id])
            
            payload = None
            for r in dense_results + sparse_results:
                if r["id"] == doc_id:
                    payload = r.get("payload", {})
                    break
            
            fused.append({
                "id": doc_id,
                "score": rank_score,
                "payload": payload,
            })
        
        fused.sort(key=lambda x: x["score"], reverse=True)
        return fused
    
    def retrieve(self, query: str, top_k: int = 0) -> list[dict]:
        """Retrieve using hybrid fusion."""
        k = top_k or self.top_k
        
        dense_results = self.dense_retriever.retrieve(query, top_k=k * 3)
        sparse_results = self.sparse_retriever.retrieve(query, top_k=k * 3)
        
        if self.fusion_method == "weighted_sum":
            fused = self._weighted_sum_fusion(dense_results, sparse_results)
        elif self.fusion_method == "rrf":
            fused = self._rrf_fusion(dense_results, sparse_results)
        elif self.fusion_method == "rank_fusion":
            fused = self._rank_fusion_with_weights(dense_results, sparse_results)
        else:
            raise ValueError(f"Unknown fusion method: {self.fusion_method}")
        
        return fused[:k]



logger.info("\n" + "=" * 60)
logger.info("EVALUATING HYBRID RETRIEVERS WITH RANK FUSION")
logger.info("=" * 60)

qdrant_client = qdrant_manager.client
TOP_K = 7
all_retriever_metrics = {}

dense_retriever = EnhancedDenseRetriever(
    collection_name=EXPERIMENT_NAME,
    embedding_model_name=EMBEDDING_MODELS[DEFAULT_EMBEDDING]["model"],
    vector_size=EMBEDDING_MODELS[DEFAULT_EMBEDDING]["vector_size"],
    top_k=TOP_K * 3,
    qdrant_client=qdrant_client,
)

2026-05-09 10:09:20,044 - INFO - 
2026-05-09 10:09:20,045 - INFO - EVALUATING HYBRID RETRIEVERS WITH RANK FUSION
2026-05-09 10:09:20,047 - INFO - ============================================================
2026-05-09 10:09:20,047 - INFO - Loading embedding model: BAAI/bge-m3
2026-05-09 10:09:20,052 - INFO - Use pytorch device_name: cuda:0
2026-05-09 10:09:20,053 - INFO - Load pretrained SentenceTransformer: BAAI/bge-m3


In [40]:
TOP_K = 7
logger.info("\n--- Testing Weighted Sum Fusion ---")
hybrid_weighted = HybridRetriever(
    dense_retriever=dense_retriever,
    sparse_retriever=tfidf_retriever,
    fusion_method="weighted_sum",
    dense_weight=0.2,
    sparse_weight=0.8,
    normalize_scores=True,
    top_k=TOP_K,
)

weighted_metrics = evaluate_retriever(
    all_samples, hybrid_weighted, TOP_K, "Hybrid (Weighted Sum)"
)
all_retriever_metrics["hybrid_weighted_sum"] = weighted_metrics

2026-05-09 10:10:08,348 - INFO - 
--- Testing Weighted Sum Fusion ---
0it [00:00, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

1it [00:00,  3.67it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

2it [00:00,  4.43it/s]

...

124it [00:22,  5.64it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

125it [00:22,  5.60it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

126it [00:22,  5.54it/s]
2026-05-09 10:10:31,087 - INFO - 
Hybrid (Weighted Sum) Metrics (Top-7):
2026-05-09 10:10:31,088 - INFO -   precision_at_k: 0.4819
2026-05-09 10:10:31,088 - INFO -   recall_at_k: 0.6861
2026-05-09 10:10:31,089 - INFO -   accuracy_at_k: 0.9127
2026-05-09 10:10:31,090 - INFO -   f1_at_k: 0.5653
2026-05-09 10:10:31,090 - INFO -   mrr_at_k: 0.8431
2026-05-09 10:10:31,091 - INFO -   ndcg_at_k: 0.8358


In [41]:
logger.info("\n--- Testing Standard RRF ---")
hybrid_rrf = HybridRetriever(
    dense_retriever=dense_retriever,
    sparse_retriever=tfidf_retriever,
    fusion_method="rrf",
    rrf_k=60,
    top_k=TOP_K,
)

rrf_metrics = evaluate_retriever(
    all_samples, hybrid_rrf, TOP_K, "Hybrid (RRF - k=60)"
)
all_retriever_metrics["hybrid_rrf"] = rrf_metrics

2026-05-09 10:10:31,098 - INFO - 
--- Testing Standard RRF ---
0it [00:00, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

1it [00:00,  5.61it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

2it [00:00,  5.62it/s]

...

124it [00:22,  5.77it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

125it [00:22,  5.80it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

126it [00:22,  5.60it/s]
2026-05-09 10:10:53,590 - INFO - 
Hybrid (RRF - k=60) Metrics (Top-7):
2026-05-09 10:10:53,591 - INFO -   precision_at_k: 0.4751
2026-05-09 10:10:53,591 - INFO -   recall_at_k: 0.6770
2026-05-09 10:10:53,592 - INFO -   accuracy_at_k: 0.9127
2026-05-09 10:10:53,592 - INFO -   f1_at_k: 0.5575
2026-05-09 10:10:53,593 - INFO -   mrr_at_k: 0.8398
2026-05-09 10:10:53,593 - INFO -   ndcg_at_k: 0.8323


In [42]:
logger.info("\n--- Testing Weighted Rank Fusion ---")
hybrid_rank_weighted = HybridRetriever(
    dense_retriever=dense_retriever,
    sparse_retriever=tfidf_retriever,
    fusion_method="rank_fusion",
    dense_weight=0.4,
    sparse_weight=0.6,
    rrf_k=30,
    top_k=TOP_K,
)

rank_weighted_metrics = evaluate_retriever(
    all_samples, hybrid_rank_weighted, TOP_K, "Hybrid (Weighted Rank Fusion)"
)
all_retriever_metrics["hybrid_rank_weighted"] = rank_weighted_metrics

2026-05-09 10:10:53,602 - INFO - 
--- Testing Weighted Rank Fusion ---
0it [00:00, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

1it [00:00,  5.66it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

2it [00:00,  5.62it/s]

...

124it [00:21,  5.66it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

125it [00:22,  5.74it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

126it [00:22,  5.66it/s]
2026-05-09 10:11:15,877 - INFO - 
Hybrid (Weighted Rank Fusion) Metrics (Top-7):
2026-05-09 10:11:15,878 - INFO -   precision_at_k: 0.4875
2026-05-09 10:11:15,878 - INFO -   recall_at_k: 0.6948
2026-05-09 10:11:15,879 - INFO -   accuracy_at_k: 0.9048
2026-05-09 10:11:15,879 - INFO -   f1_at_k: 0.5722
2026-05-09 10:11:15,880 - INFO -   mrr_at_k: 0.8499
2026-05-09 10:11:15,881 - INFO -   ndcg_at_k: 0.8377


In [ ]:

from transformers import (
    AutoTokenizer, 
    AutoModelForSequenceClassification,
    BitsAndBytesConfig,
    AutoModelForCausalLM
)

class HybridRetrieverWithRerank:
    """Hybrid retriever with optional re-ranking after fusion."""
    
    def __init__(
        self,
        dense_retriever,
        sparse_retriever,
        reranker=None,
        fusion_method: str = "weighted_sum",
        dense_weight: float = 0.7,
        sparse_weight: float = 0.3,
        normalize_scores: bool = True,
        rrf_k: int = 60,
        top_k: int = 7,
        rerank_top_k: int = 20,
    ):
        self.dense_retriever = dense_retriever
        self.sparse_retriever = sparse_retriever
        self.reranker = reranker
        self.fusion_method = fusion_method
        self.dense_weight = dense_weight
        self.sparse_weight = sparse_weight
        self.normalize_scores = normalize_scores
        self.rrf_k = rrf_k
        self.top_k = top_k
        self.rerank_top_k = rerank_top_k
    
    def _normalize_scores(self, results: list[dict]) -> list[dict]:
        """Min-max normalize scores to [0, 1] range."""
        if not results:
            return results
        
        scores = [r["score"] for r in results]
        min_score = min(scores)
        max_score = max(scores)
        
        if max_score == min_score:
            for r in results:
                r["normalized_score"] = 0.5
            return results
        
        for r in results:
            r["normalized_score"] = (r["score"] - min_score) / (max_score - min_score)
        
        return results
    
    def _weighted_sum_fusion(self, dense_results, sparse_results) -> list[dict]:
        """Fuse scores using weighted sum of normalized scores."""
        if self.normalize_scores:
            dense_results = self._normalize_scores(dense_results)
            sparse_results = self._normalize_scores(sparse_results)
            dense_key = "normalized_score"
            sparse_key = "normalized_score"
        else:
            dense_key = "score"
            sparse_key = "score"
        
        dense_dict = {r["id"]: r[dense_key] for r in dense_results}
        sparse_dict = {r["id"]: r[sparse_key] for r in sparse_results}
        
        all_ids = set(dense_dict.keys()) | set(sparse_dict.keys())
        
        fused = []
        for doc_id in all_ids:
            dense_score = dense_dict.get(doc_id, 0) * self.dense_weight
            sparse_score = sparse_dict.get(doc_id, 0) * self.sparse_weight
            fused_score = dense_score + sparse_score
            
            payload = None
            for r in dense_results + sparse_results:
                if r["id"] == doc_id:
                    payload = r.get("payload", {})
                    break
            
            fused.append({
                "id": doc_id,
                "score": fused_score,
                "payload": payload,
                "dense_score": dense_dict.get(doc_id, 0),
                "sparse_score": sparse_dict.get(doc_id, 0),
            })
        
        fused.sort(key=lambda x: x["score"], reverse=True)
        return fused
    
    def _rrf_fusion(self, dense_results, sparse_results) -> list[dict]:
        """Reciprocal Rank Fusion (RRF)."""
        dense_ranks = {r["id"]: idx + 1 for idx, r in enumerate(dense_results)}
        sparse_ranks = {r["id"]: idx + 1 for idx, r in enumerate(sparse_results)}
        
        all_ids = set(dense_ranks.keys()) | set(sparse_ranks.keys())
        
        fused = []
        for doc_id in all_ids:
            rrf_score = 0.0
            
            if doc_id in dense_ranks:
                rrf_score += 1.0 / (self.rrf_k + dense_ranks[doc_id])
            if doc_id in sparse_ranks:
                rrf_score += 1.0 / (self.rrf_k + sparse_ranks[doc_id])
            
            payload = None
            for r in dense_results + sparse_results:
                if r["id"] == doc_id:
                    payload = r.get("payload", {})
                    break
            
            fused.append({
                "id": doc_id,
                "score": rrf_score,
                "payload": payload,
                "dense_rank": dense_ranks.get(doc_id, None),
                "sparse_rank": sparse_ranks.get(doc_id, None),
            })
        
        fused.sort(key=lambda x: x["score"], reverse=True)
        return fused
    
    def _rank_weighted_fusion(self, dense_results, sparse_results) -> list[dict]:
        """Weighted Rank Fusion."""
        dense_ranks = {r["id"]: idx + 1 for idx, r in enumerate(dense_results)}
        sparse_ranks = {r["id"]: idx + 1 for idx, r in enumerate(sparse_results)}
        
        all_ids = set(dense_ranks.keys()) | set(sparse_ranks.keys())
        
        fused = []
        for doc_id in all_ids:
            rank_score = 0.0
            
            if doc_id in dense_ranks:
                rank_score += self.dense_weight / (self.rrf_k + dense_ranks[doc_id])
            if doc_id in sparse_ranks:
                rank_score += self.sparse_weight / (self.rrf_k + sparse_ranks[doc_id])
            
            payload = None
            for r in dense_results + sparse_results:
                if r["id"] == doc_id:
                    payload = r.get("payload", {})
                    break
            
            fused.append({
                "id": doc_id,
                "score": rank_score,
                "payload": payload,
            })
        
        fused.sort(key=lambda x: x["score"], reverse=True)
        return fused
    
    def retrieve(self, query: str, top_k: int = 0) -> list[dict]:
        """Retrieve using hybrid fusion with optional re-ranking."""
        k = top_k or self.top_k
        
        candidate_k = self.rerank_top_k if self.reranker else k * 3
        dense_results = self.dense_retriever.retrieve(query, top_k=candidate_k)
        sparse_results = self.sparse_retriever.retrieve(query, top_k=candidate_k)
        
        if self.fusion_method == "weighted_sum":
            fused = self._weighted_sum_fusion(dense_results, sparse_results)
        elif self.fusion_method == "rrf":
            fused = self._rrf_fusion(dense_results, sparse_results)
        elif self.fusion_method == "rank_fusion":
            fused = self._rank_weighted_fusion(dense_results, sparse_results)
        else:
            raise ValueError(f"Unknown fusion method: {self.fusion_method}")
        
        if self.reranker and self.rerank_top_k > 0:
            candidates_to_rerank = fused[:self.rerank_top_k]
            reranked = self.reranker.rerank(query, candidates_to_rerank, top_k=k)
            return reranked
        else:
            return fused[:k]



class EfficientCrossEncoderReranker:
    """Correctly handles cross-encoder outputs."""
    
    def __init__(self, model_name: str = "BAAI/bge-reranker-large",
                 max_length: int = 512, device: str = "cuda"):
        self.model_name = model_name
        self.max_length = max_length
        self.device = device
        
        print(f"Loading cross-encoder model: {model_name}")
        
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModelForSequenceClassification.from_pretrained(
            model_name,
            torch_dtype=torch.float16 if device == "cuda" else torch.float32,
        )
        self.model = self.model.to(device)
        self.model.eval()
        
        self.num_labels = self.model.config.num_labels
        print(f"Model has {self.num_labels} output classes")
        
        self.inversion_detected = False
        self.sample_scores = []
    
    def rerank(self, query: str, documents: list[dict], top_k: int = None) -> list[dict]:
        if not documents:
            return []
        
        pairs = []
        valid_docs = []
        
        for doc in documents:
            text = doc.get("payload", {}).get("text", "")
            if text:
                if len(text) > 512:
                    text = text[:512]
                pairs.append(f"{query} [SEP] {text}")
                valid_docs.append(doc)
        
        if not pairs:
            return documents[:top_k] if top_k else documents
        
        batch_size = 16
        all_scores = []
        
        for i in range(0, len(pairs), batch_size):
            batch_pairs = pairs[i:i+batch_size]
            
            inputs = self.tokenizer(
                batch_pairs,
                truncation=True,
                padding=True,
                max_length=self.max_length,
                return_tensors="pt"
            )
            inputs = {k: v.to(self.device) for k, v in inputs.items()}
            
            with torch.no_grad():
                outputs = self.model(**inputs)
                
                logits = outputs.logits.squeeze(-1)
                
                scores = torch.sigmoid(logits).cpu().tolist()
                
                if i == 0 and not self.inversion_detected:
                    print(f"Sample scores: {[f'{s:.4f}' for s in scores[:3]]}")
                    
                    if len(scores) > 0 and scores[0] < 0.3:
                        print(f"⚠️ Possible score inversion detected! Score={scores[0]:.4f}")
                        self.inversion_detected = True
                
                all_scores.extend(scores)
        
        if self.inversion_detected:
            all_scores = [1.0 - s for s in all_scores]
            print("✅ Inverted scores applied")
        
        scored_docs = []
        for doc, score in zip(valid_docs, all_scores):
            reranked_doc = doc.copy()
            reranked_doc["score"] = score
            reranked_doc["rerank_score"] = score
            scored_docs.append(reranked_doc)
        
        scored_docs.sort(key=lambda x: x["rerank_score"], reverse=True)
        
        k = top_k if top_k is not None else len(scored_docs)
        return scored_docs[:k]



logger.info("=" * 60)
logger.info("EVALUATING HYBRID RETRIEVERS WITH RE-RANKING")
logger.info("=" * 60)

reranker = EfficientCrossEncoderReranker(device="cuda" if torch.cuda.is_available() else "cpu")

TOP_K = 7
all_retriever_metrics = {}

logger.info("\n--- Hybrid Weighted Sum (No Rerank) ---")
hybrid_weighted = HybridRetrieverWithRerank(
    dense_retriever=dense_retriever,
    sparse_retriever=tfidf_retriever,
    reranker=None,
    fusion_method="weighted_sum",
    dense_weight=0.7,
    sparse_weight=0.3,
    top_k=TOP_K,
)
weighted_metrics = evaluate_retriever(all_samples, hybrid_weighted, TOP_K, "Hybrid Weighted Sum")
all_retriever_metrics["hybrid_weighted_sum"] = weighted_metrics

logger.info("\n--- Hybrid Weighted Sum + Reranker ---")
hybrid_weighted_rerank = HybridRetrieverWithRerank(
    dense_retriever=dense_retriever,
    sparse_retriever=tfidf_retriever,
    reranker=reranker,
    fusion_method="weighted_sum",
    dense_weight=0.7,
    sparse_weight=0.3,
    rerank_top_k=20,
    top_k=TOP_K,
)
weighted_rerank_metrics = evaluate_retriever(all_samples, hybrid_weighted_rerank, TOP_K, "Hybrid Weighted Sum + Rerank")
all_retriever_metrics["hybrid_weighted_sum_rerank"] = weighted_rerank_metrics

---
## Cell 9: Generator Evaluation

In [129]:

class QwenGenerator:
    """Qwen-based answer generator with 4-bit quantization support."""

    def __init__(self, model_name: str, device: str = "cuda"):
        self.model_name = model_name
        
        load_in_4bit = True
        
        if load_in_4bit:
            from transformers import BitsAndBytesConfig
            
            logger.info(f"Loading {model_name} with 4-bit quantization")
            bnb_config = BitsAndBytesConfig(
                load_in_4bit=True,
                bnb_4bit_compute_dtype=torch.float16,
                bnb_4bit_quant_type="nf4",
                bnb_4bit_use_double_quant=True,
                llm_int8_enable_fp32_cpu_offload=True,
            )
            self.tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
            self.model = AutoModelForCausalLM.from_pretrained(
                model_name,
                quantization_config=bnb_config,
                device_map="auto",
                trust_remote_code=True,
            )
        else:
            logger.info(f"Loading {model_name} without quantization")
            self.tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
            self.model = AutoModelForCausalLM.from_pretrained(
                model_name,
                torch_dtype=torch.float16 if device == "cuda" else torch.float32,
                device_map=device,
                trust_remote_code=True,
            )
        
        self.model.eval()
    
    def generate(self, query: str, context: str, max_new_tokens: int = 256) -> str:
        prompt = f"Context:\n{context}\n\nQuestion: {query}\nAnswer:"
        
        messages = [
            {
                "role": "system",
                "content": "You are a helpful scientific assistant. Answer based ONLY on the provided context.",
            },
            {"role": "user", "content": prompt},
        ]
        
        text = self.tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True
        )
        inputs = self.tokenizer(text, return_tensors="pt").to(self.model.device)
        
        with torch.no_grad():
            outputs = self.model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                do_sample=False,
                pad_token_id=self.tokenizer.eos_token_id,
            )
        
        response = self.tokenizer.decode(
            outputs[0][inputs.input_ids.shape[1] :], skip_special_tokens=True
        )
        return response.strip()


def compute_rouge(predictions: list[str], references: list[str]) -> float:
    scorer = rouge_scorer.RougeScorer(["rougeL"], use_stemmer=True)
    scores = [scorer.score(ref, pred) for ref, pred in zip(references, predictions, strict=False)]
    return sum(s["rougeL"].fmeasure for s in scores) / len(scores)


def compute_bleu(predictions: list[str], references: list[str]) -> float:
    refs = [[ref] for ref in references]
    bleu = sacrebleu.corpus_bleu(predictions, refs)
    return bleu.score / 100.0


def compute_bertscore(predictions: list[str], references: list[str], model_type: str) -> float:
    _, _, f1 = bert_score(predictions, references, model_type=model_type, lang="en", verbose=False)
    return f1.mean().item()


def compute_faithfulness(answers: list[str], contexts: list[str]) -> float:
    scores = []
    for ans, ctx in zip(answers, contexts, strict=False):
        if not ans.strip():
            scores.append(0.0)
            continue
        vec = CountVectorizer(ngram_range=(1, 1), lowercase=True, token_pattern=r"\b\w+\b")
        try:
            vec.fit([ctx])
            ctx_vocab = set(vec.get_feature_names_out())
            ans_tokens = vec.build_analyzer()(ans)
            if not ans_tokens:
                scores.append(0.0)
                continue
            in_context = sum(1 for token in ans_tokens if token in ctx_vocab)
            scores.append(in_context / len(ans_tokens))
        except ValueError:
            scores.append(0.0)
    return sum(scores) / len(scores) if scores else 0.0


print("✅ Generator evaluation functions defined")

✅ Generator evaluation functions defined


---
## Cell 10: LLM-as-a-Judge (Optional)

In [130]:

MAX_SCORE = 5

EVALUATION_PROMPTS = {
    "answer_relevance": """You are an expert evaluator for question-answering systems.
Evaluate how well the answer addresses the question.

IMPORTANT: Output ONLY a valid JSON object with exactly this key:
"score": <integer between 1 and 5>
Do not include any additional text, reasoning, or explanations outside the JSON.

Question: {question}
Answer: {answer}
""",
    "faithfulness": """You are an expert evaluator for faithfulness/grounding.
Verify whether all claims in the answer are supported by the provided context.
IMPORTANT: Output ONLY a valid JSON object with exactly this key:
"score": <integer between 1 and 5>
Do not include any additional text, reasoning, or explanations outside the JSON.

Context: {context}
Answer: {answer}
""",
    "context_relevance": """You are an expert evaluator for context relevance in question-answering systems.
Evaluate how relevant and complete the retrieved context is for answering the question.
Consider both whether the context contains the information needed to answer the question
and whether it provides sufficient detail and completeness.

IMPORTANT: Output ONLY a valid JSON object with exactly this key:
"score": <integer between 1 and 5>
Do not include any additional text, reasoning, or explanations outside the JSON.

Question: {question}
Context: {context}
""",
}


def evaluate_with_llm_judge(
    questions: list[str],
    predictions: list[str],
    ground_truths: list[str],
    contexts: list[str],
    judge_pipe,
) -> dict:
    """Evaluate predictions using LLM-as-a-judge with RAG triad metrics."""
    metrics = ["answer_relevance", "faithfulness", "context_relevance"]
    results = {metric: [] for metric in metrics}
    
    logger.info(f"Evaluating {len(questions)} samples with LLM-as-a-Judge (RAG Triad)...")
    
    for i, (question, prediction, ground_truth, context) in enumerate(
        zip(questions, predictions, ground_truths, contexts, strict=False)
    ):
        for metric in metrics:
            if metric == "faithfulness" and not context:
                continue
            if metric == "context_relevance" and not context:
                continue
            
            prompt_templates = {
                "answer_relevance": EVALUATION_PROMPTS["answer_relevance"].format(
                    question=question, answer=prediction
                ),
                "faithfulness": EVALUATION_PROMPTS["faithfulness"].format(
                    context=context, answer=prediction
                ),
                "context_relevance": EVALUATION_PROMPTS["context_relevance"].format(
                    question=question, context=context
                ),
            }
            
            prompt = prompt_templates[metric]
            
            try:
                messages = [{"role": "user", "content": prompt}]
                output = judge_pipe(
                    messages,
                    max_new_tokens=100,
                    temperature=0.0,
                    do_sample=False,
                )
                
                content = output[0]["generated_text"]
                if isinstance(content, list):
                    content = content[-1]["content"]
                
                json_str = extract_json_from_text(content)
                if not json_str:
                    continue
                
                parsed = json.loads(json_str)
                score = parsed.get("score")
                
                if score is not None and 1 <= score <= MAX_SCORE:
                    results[metric].append(score / MAX_SCORE)
            except Exception as e:
                logger.warning(f"Failed to evaluate {metric} for sample {i}: {e}")
                continue
        
        logger.info(f"Processed {i + 1}/{len(questions)} samples")
    
    averages = {}
    for metric in metrics:
        scores = results[metric]
        averages[metric] = sum(scores) / len(scores) if scores else 0.0
    
    logger.info("LLM-as-a-Judge Results (RAG Triad):")
    for metric, avg_score in averages.items():
        logger.info(f"  {metric}: {avg_score:.4f}")
    
    return averages


print("✅ LLM-as-a-Judge functions defined (RAG Triad: answer_relevance, faithfulness, context_relevance)")

✅ LLM-as-a-Judge functions defined (RAG Triad: answer_relevance, faithfulness, context_relevance)


In [134]:
import gc
import torch
import time

del generator, test_data_pipe, _model, _tokenizer

gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()
    print(f"✅ GPU memory cleared. Allocated: {torch.cuda.memory_allocated(0)/1e9:.2f} GB")

✅ GPU memory cleared. Allocated: 2.28 GB


---
## Cell 11: Run Generator Evaluation

In [ ]:
import time
import statistics
from typing import Dict, List, Tuple
import matplotlib.pyplot as plt
import pandas as pd
from pathlib import Path

class QwenGeneratorWithLatency(QwenGenerator):
    """QwenGenerator with latency measurement capabilities."""
    
    def __init__(self, model_name: str, device: str = "cuda"):
        super().__init__(model_name, device)
        self.latency_records = []
        self.tokens_generated = []

        if hasattr(self.tokenizer, "chat_template") and self.tokenizer.chat_template is None:
            self.tokenizer.chat_template = """{% for message in messages %}{% if loop.first and messages[0]['role'] != 'system' %}{{ '<|im_start|>system\nYou are a helpful assistant.<|im_end|>\n' }}{% endif %}{{'<|im_start|>' + message['role'] + '\n' + message['content'] + '<|im_end|>' + '\n'}}{% endfor %}{% if add_generation_prompt %}{{ '<|im_start|>assistant\n' }}{% endif %}"""
            logger.info("✅ Set Qwen chat template manually")
    
    def generate_with_latency(self, query: str, context: str, max_new_tokens: int = 256) -> Tuple[str, Dict]:
        """Generate answer with latency measurement."""
        
        prompt = f"Context:\n{context}\n\nQuestion: {query}\nAnswer:"
        
        messages = [
            {
                "role": "system",
                "content": "You are a helpful scientific assistant. Answer based ONLY on the provided context.",
            },
            {"role": "user", "content": prompt},
        ]
        
        text = self.tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True
        )
        inputs = self.tokenizer(text, return_tensors="pt").to(self.model.device)
        input_length = inputs.input_ids.shape[1]
        
        start_time = time.perf_counter()
        
        with torch.no_grad():
            outputs = self.model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                do_sample=False,
                pad_token_id=self.tokenizer.eos_token_id,
            )
        
        end_time = time.perf_counter()
        
        total_latency = end_time - start_time
        output_ids = outputs[0][input_length:]
        num_tokens = len(output_ids)
        tokens_per_second = num_tokens / total_latency if total_latency > 0 else 0
        
        response = self.tokenizer.decode(output_ids, skip_special_tokens=True)
        
        latency_info = {
            "total_latency_sec": total_latency,
            "num_tokens": num_tokens,
            "tokens_per_second": tokens_per_second,
            "input_length": input_length,
        }
        
        self.latency_records.append(total_latency)
        self.tokens_generated.append(num_tokens)
        
        return response.strip(), latency_info
    
    def get_latency_stats(self) -> Dict:
        """Get aggregated latency statistics."""
        if not self.latency_records:
            return {}
        
        return {
            "mean_latency_sec": statistics.mean(self.latency_records),
            "median_latency_sec": statistics.median(self.latency_records),
            "std_latency_sec": statistics.stdev(self.latency_records) if len(self.latency_records) > 1 else 0,
            "min_latency_sec": min(self.latency_records),
            "max_latency_sec": max(self.latency_records),
            "total_requests": len(self.latency_records),
            "mean_tokens": statistics.mean(self.tokens_generated),
            "mean_tokens_per_second": statistics.mean([t/l for t, l in zip(self.tokens_generated, self.latency_records)]),
        }
    
    def reset_latency_records(self):
        """Reset latency records for new evaluation."""
        self.latency_records = []
        self.tokens_generated = []


print("✅ QwenGeneratorWithLatency class defined")

✅ QwenGeneratorWithLatency class defined


In [ ]:
logger.info("=" * 60)
logger.info("STEP 6: GENERATOR EVALUATION WITH LATENCY")
logger.info("=" * 60)

generator_retriever = dense_retriever

GENERATOR_MODEL = "unsloth/Llama-3.2-3B-Instruct"
logger.info(f"Loading generator model: {GENERATOR_MODEL}")
generator = QwenGeneratorWithLatency(GENERATOR_MODEL, device="cuda" if torch.cuda.is_available() else "cpu")
logger.info(f"✅ Generator loaded")

predictions = []
references = []
contexts = []
questions_list = []
latency_data = []

logger.info(f"Evaluating on {len(all_samples)} test samples...")

for i, sample in enumerate(all_samples[:5]):
    logger.info(f"\rGenerating {i + 1}/{len(all_samples)}")
    
    docs = generator_retriever.retrieve(sample["question"], top_k=TOP_K)
    context = "\n\n".join([doc["payload"]["text"] for doc in docs])
    
    pred, latency_info = generator.generate_with_latency(
        sample["question"], 
        context, 
        max_new_tokens=GENERATOR_MAX_TOKENS
    )
    
    predictions.append(pred)
    references.append(sample["answer"])
    contexts.append(context)
    questions_list.append(sample["question"])
    latency_data.append(latency_info)

logger.info("\n✅ Generation complete!")

logger.info("Computing traditional metrics...")
rouge = compute_rouge(predictions, references)
bleu = compute_bleu(predictions, references)
bert_f1 = compute_bertscore(predictions, references, BERTSCORE_MODEL)
faithfulness = compute_faithfulness(predictions, contexts)

generator_metrics = {
    "rougeL": rouge,
    "bleu": bleu,
    "bertscore_f1": bert_f1,
    "faithfulness_ngram": faithfulness,
}

logger.info("\nGenerator Metrics (Traditional):")
for k, v in generator_metrics.items():
    logger.info(f"  {k}: {v:.4f}")

latency_stats = generator.get_latency_stats()
logger.info("\nLatency Statistics:")
for k, v in latency_stats.items():
    if "latency" in k:
        logger.info(f"  {k}: {v:.4f} sec")
    elif "tokens" in k:
        logger.info(f"  {k}: {v:.2f}")
    else:
        logger.info(f"  {k}: {v}")

del generator
time.sleep(15)
gc.collect()

logger.info("✅ Generator evaluation complete!")

2026-05-09 11:06:22,715 - INFO - ============================================================
2026-05-09 11:06:22,717 - INFO - STEP 6: GENERATOR EVALUATION WITH LATENCY
2026-05-09 11:06:22,718 - INFO - ============================================================
2026-05-09 11:06:22,719 - INFO - Loading generator model: unsloth/Llama-3.2-3B-Instruct
2026-05-09 11:06:22,720 - INFO - Loading unsloth/Llama-3.2-3B-Instruct with 4-bit quantization


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/454 [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

...

Generating 4/126:02,539 - INFO - 


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Generating 5/126:18,029 - INFO - 


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

2026-05-09 11:08:39,410 - INFO - 
✅ Generation complete!
2026-05-09 11:08:39,412 - INFO - Computing traditional metrics...
2026-05-09 11:08:39,413 - INFO - Using default tokenizer.
Some weights of RobertaModel were not initialized from the model checkpoint at roberta-base and are newly initialized: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
2026-05-09 11:08:41,629 - INFO - 
Generator Metrics (Traditional):
2026-05-09 11:08:41,630 - INFO -   rougeL: 0.4146
2026-05-09 11:08:41,630 - INFO -   bleu: 0.1884
2026-05-09 11:08:41,631 - INFO -   bertscore_f1: 0.8870
2026-05-09 11:08:41,632 - INFO -   faithfulness_ngram: 0.9816
2026-05-09 11:08:41,633 - INFO - 
Latency Statistics:
2026-05-09 11:08:41,634 - INFO -   mean_latency_sec: 15.5291 sec
2026-05-09 11:08:41,634 - INFO -   median_latency_sec: 15.3012 sec
2026-05-09 11:08:41,635 - INFO -   std_latency_sec: 4.1194 

In [ ]:

logger.info("=" * 60)
logger.info("STEP 6: GENERATOR EVALUATION")
logger.info("=" * 60)

best_retriever_name = "dense_rerank"
best_retriever = eval(best_retriever_name)

generator_retriever = dense_retriever

logger.info(f"Loading generator model: {GENERATOR_MODEL}")
generator = QwenGenerator(GENERATOR_MODEL, device="cuda" if torch.cuda.is_available() else "cpu")
logger.info(f"✅ Generator loaded")

predictions = []
references = []
contexts = []
questions_list = []

for i, sample in enumerate(all_samples):
    logger.info(f"\rGenerating {i + 1}/{len(all_samples)}")
    
    docs = generator_retriever.retrieve(sample["question"], top_k=TOP_K)
    context = "\n\n".join([doc["payload"]["text"] for doc in docs])
    
    pred = generator.generate(sample["question"], context, max_new_tokens=GENERATOR_MAX_TOKENS)
    
    predictions.append(pred)
    references.append(sample["answer"])
    contexts.append(context)
    questions_list.append(sample["question"])

logger.info("Computing traditional metrics...")
rouge = compute_rouge(predictions, references)
bleu = compute_bleu(predictions, references)
bert_f1 = compute_bertscore(predictions, references, BERTSCORE_MODEL)
faithfulness = compute_faithfulness(predictions, contexts)

generator_metrics = {
    "rougeL": rouge,
    "bleu": bleu,
    "bertscore_f1": bert_f1,
    "faithfulness_ngram": faithfulness,
}

logger.info("Generator Metrics (Traditional):")
for k, v in generator_metrics.items():
    logger.info(f"  {k}: {v:.4f}")


del generator
time.sleep(15)
gc.collect()

llm_metrics = {}
if USE_LLM_JUDGE:
    logger.info("Loading LLM Judge model...")
    _judge_tokenizer = AutoTokenizer.from_pretrained(LLM_JUDGE_MODEL)
    _judge_model = AutoModelForCausalLM.from_pretrained(
        LLM_JUDGE_MODEL,
        device_map="auto",
        torch_dtype=torch.float16,
        quantization_config=bnb_config
    )
    judge_pipe = pipeline("text-generation", model=_judge_model, tokenizer=_judge_tokenizer)
    
    llm_metrics = evaluate_with_llm_judge(
        questions_list, predictions, references, contexts, judge_pipe
    )

all_generator_metrics = {**generator_metrics, **llm_metrics}

logger.info("✅ Generator evaluation complete!")

---
## Cell 12: Save Final Results

In [ ]:

logger.info("=" * 60)
logger.info("STEP 7: SAVING RESULTS")
logger.info("=" * 60)

experiment_results = {
    "experiment_name": EXPERIMENT_NAME,
    "config": {
        "chunking": {
            "chunk_size": CHUNK_SIZE,
            "chunk_overlap": CHUNK_OVERLAP,
            "chunking_type": CHUNKING_TYPE
        },
        "embeddings": {
            "models": EMBEDDING_MODELS,
            "default_model": DEFAULT_EMBEDDING,
            "vector_size": VECTOR_SIZE
        },
        "retrievers": RETRIEVER_CONFIG,
        "qdrant": {
            "collection_name": EXPERIMENT_NAME,
            "top_k": TOP_K,
            "vector_size": VECTOR_SIZE
        },
        "test_data": {
            "model": LLM_TEST_DATA_MODEL,
            "sample_percent": TEST_DATA_SIZE_PERCENT,
            "questions_per_paper": MAX_QUESTIONS_PER_PAPER
        },
        "generator": {
            "model": GENERATOR_MODEL,
            "max_tokens": GENERATOR_MAX_TOKENS
        },
        "llm_judge": {
            "enabled": USE_LLM_JUDGE,
            "model": LLM_JUDGE_MODEL if USE_LLM_JUDGE else None
        }
    },
    "metrics": {
        "retrievers": all_retriever_metrics,
        "generator": all_generator_metrics
    },
    "data_stats": {
        "total_papers": total_papers,
        "papers_used": target_papers,
        "total_chunks": len(chunks),
        "total_test_samples": len(all_samples)
    },
    "best_retriever": best_retriever_name
}

save_results(experiment_results, RESULTS_FILE)

logger.info("\n" + "=" * 60)
logger.info("EXPERIMENT COMPLETE - ENHANCED RAG SYSTEM")
logger.info("=" * 60)
logger.info(f"Experiment: {EXPERIMENT_NAME}")
logger.info(f"Results saved to: {RESULTS_FILE}")
logger.info(f"\n=== Retriever Performance Summary ===")
for retriever_name, metrics in all_retriever_metrics.items():
    logger.info(f"  {retriever_name}: F1={metrics['f1_at_k']:.4f}, Recall={metrics['recall_at_k']:.4f}, NDCG={metrics['ndcg_at_k']:.4f}")
logger.info(f"\n=== Generator Performance ===")
for k, v in all_generator_metrics.items():
    logger.info(f"  {k}: {v:.4f}")
logger.info(f"\nBest Retriever: {best_retriever_name}")
logger.info("=" * 60)

print(f"\n🎉 Experiment completed successfully!")
print(f"📊 Check {RESULTS_FILE} for detailed results")
print(f"🔍 Use the best retriever configuration: {best_retriever_name}")